# Autonomous Scientific Animation Studio — Full Pipeline (Phase 1 + Phase 2)

One Google Colab notebook that turns a **topic** into a finished, upload-ready
scientific short.

```text
Topic
  → Free web research (DDGS + Wikipedia + Crossref + OpenAlex + arXiv)
  → Deterministic scientific validation
  → Retention-first script (timed beats)
  → Storyboard JSON
  → Scene plan (asset catalogue)
  → Deterministic SVG asset generation + content-addressed cache
  → Static scene composition
  → Voice-over (Edge TTS → espeak-ng fallback)
  → Safe-zone subtitles (SRT + ASS)
  → Procedural music + sound cues, side-chain ducked & loudness-normalised
  → Animated 1080×1920 MP4 (FFmpeg zoompan) with burned-in captions
  → ffprobe self-validation
  → SEO metadata → manifest + downloadable ZIP
```

**Phase 1** = everything up to the static storyboard preview.
**Phase 2** = voice, captions, audio mix, animated MP4, and SEO.
Both run in a single `StudioPipeline.run()`.

## Engineering decisions

- **One notebook, one importable package.** Every section writes a real module
  into `src/autostudio/`; nothing lives only in a cell. `StudioPipeline.run()`
  is the single entry point and lifts into FastAPI/Docker without a rewrite.
- **Deterministic SVG.** The LLM never writes SVG — it only names an
  `asset_type` from a whitelist; Python renders the flat vector from a
  controlled template. Reproducible, editable, cacheable; no raster/gradient.
- **Cache-first, content-addressed.** Web search, LLM responses, SVG assets,
  voice clips, rasters, and rendered scene clips are all keyed by content hash.
  Re-running an identical stage is a cache hit, never a regeneration.
- **Local-first models.** Qwen (HuggingFace) primary; OpenAI optional fallback
  (used only if a key exists and local fails). No API is hardcoded.
- **Free-only audio/video.** Edge TTS + espeak-ng for voice; FFmpeg `lavfi`
  synthesises the music bed and sound cues — no external audio assets or
  licensing. CairoSVG + FFmpeg do all rasterization and rendering.
- **Low GPU usage.** The GPU is used only by Qwen. Search, SVG, TTS
  orchestration, mixing, and FFmpeg all run on CPU.
- **Self-validating output.** The final MP4 is probed with `ffprobe` for
  resolution, fps, duration, and audio stream before the run is declared done.
- **Graceful degradation.** Every generative stage has a deterministic fallback,
  so a flaky network never aborts a run; the media path has a fully offline test.

## 1. Environment

Detect Colab, lay out the project folders, and put the `autostudio` package on `sys.path`. Set `USE_GOOGLE_DRIVE = True` for a persistent cache + asset library on Drive.

In [ ]:
from __future__ import annotations

import os
import sys
import platform
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = False  # Set True for a persistent cache/asset library on Drive.

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/autonomous_scientific_animation_studio")
elif IN_COLAB:
    PROJECT_ROOT = Path("/content/autonomous_scientific_animation_studio")
else:
    PROJECT_ROOT = Path.cwd() / "autonomous_scientific_animation_studio"

PROJECT_ROOT = PROJECT_ROOT.resolve()
SRC_ROOT = PROJECT_ROOT / "src"
PACKAGE_ROOT = SRC_ROOT / "autostudio"

DIRECTORIES = [
    "cache", "cache/assets", "cache/research", "cache/scripts", "cache/storyboards", "cache/llm",
    "cache/search", "cache/audio", "cache/raster", "cache/scene_clips", "cache/metadata",
    "assets", "storyboards", "scenes", "scripts", "research", "audio", "captions", "video",
    "metadata", "output", "logs", "config", "tests", "src/autostudio",
]
for relative in DIRECTORIES:
    (PROJECT_ROOT / relative).mkdir(parents=True, exist_ok=True)

(PACKAGE_ROOT / "__init__.py").write_text(
    '"""Autonomous Scientific Animation Studio."""\n__version__ = "1.0.0"\n', encoding="utf-8",
)
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
os.environ["AUTOSTUDIO_ROOT"] = str(PROJECT_ROOT)

print("Environment:", "Google Colab" if IN_COLAB else "Local Jupyter")
print("Python:", platform.python_version())
print("Project root:", PROJECT_ROOT)

## 2. Dependency installation

Installs the full stack: models, free web search, SVG, and the audio/video toolchain (FFmpeg, Cairo, Edge TTS + espeak-ng). On a local machine the `apt-get` line is skipped automatically.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    get_ipython().system('apt-get -qq update')
    get_ipython().system('apt-get -qq install -y ffmpeg espeak-ng libcairo2 libpango-1.0-0 fonts-dejavu-core')

%pip install -q \
  "transformers>=4.48,<5" \
  "accelerate>=1.2,<2" \
  "bitsandbytes>=0.45,<1" \
  "pydantic>=2.10,<3" \
  "PyYAML>=6,<7" \
  "requests>=2.32,<3" \
  "ddgs>=9,<10" \
  "feedparser>=6,<7" \
  "tenacity>=9,<10" \
  "tqdm>=4.67,<5" \
  "svgwrite>=1.4,<2" \
  "CairoSVG>=2.7,<3" \
  "Pillow>=10,<13" \
  "edge-tts>=7,<8" \
  "nest-asyncio>=1.6,<2" \
  "openai>=1.60,<3"

print("Dependencies installed.")

## 3. Configuration, schemas, hashing, and logging

The foundation layer — typed Pydantic schemas for every artifact (through audio,
captions, render report, and SEO), a YAML-backed `StudioConfig` covering the LLM,
GPU/quantization, canvas, palette, typography, voice, captions, audio mix, and
render settings, plus content-hashing + atomic writes and one idempotent logger.

In [ ]:
# Section 3 — hashing, schemas, config, logging
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "hashing.py": "from __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport re\nimport tempfile\nfrom pathlib import Path\nfrom typing import Any\n\n\ndef canonical_json(value: Any) -> str:\n    \"\"\"Deterministic, stable JSON encoding used for every content hash.\"\"\"\n    if hasattr(value, \"model_dump\"):\n        value = value.model_dump(mode=\"json\")\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\",\", \":\"), default=str)\n\n\ndef hash_value(value: Any) -> str:\n    \"\"\"Content hash for any JSON-serialisable object (research/script/asset keys).\"\"\"\n    return hashlib.sha256(canonical_json(value).encode(\"utf-8\")).hexdigest()\n\n\ndef file_sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef slugify(value: str, max_length: int = 80) -> str:\n    value = re.sub(r\"[^a-zA-Z0-9]+\", \"-\", value).strip(\"-\").lower()\n    return (value or \"untitled\")[:max_length]\n\n\ndef atomic_write_text(path: Path, content: str) -> None:\n    \"\"\"Crash-safe write: never leaves a half-written file in the cache.\"\"\"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temp_name = tempfile.mkstemp(prefix=path.name, suffix=\".tmp\", dir=str(path.parent))\n    try:\n        with os.fdopen(fd, \"w\", encoding=\"utf-8\") as handle:\n            handle.write(content)\n        os.replace(temp_name, path)\n    finally:\n        if os.path.exists(temp_name):\n            os.unlink(temp_name)\n\n\ndef atomic_write_json(path: Path, value: Any) -> None:\n    if hasattr(value, \"model_dump\"):\n        value = value.model_dump(mode=\"json\")\n    atomic_write_text(path, json.dumps(value, ensure_ascii=False, indent=2, default=str))\n\n\ndef read_json(path: Path, default: Any = None) -> Any:\n    return json.loads(path.read_text(encoding=\"utf-8\")) if path.exists() else default\n",
    "schemas.py": "from __future__ import annotations\n\nfrom datetime import datetime, timezone\nfrom typing import Any\n\nfrom pydantic import BaseModel, ConfigDict, Field\n\n\nclass StudioModel(BaseModel):\n    \"\"\"Base model. `extra=\"ignore\"` keeps LLM output tolerant; assignment is validated.\"\"\"\n\n    model_config = ConfigDict(extra=\"ignore\", validate_assignment=True)\n\n\n# --------------------------------------------------------------------------- #\n# Search + research\n# --------------------------------------------------------------------------- #\nclass SourceRecord(StudioModel):\n    source_id: str\n    provider: str\n    title: str\n    url: str | None = None\n    snippet: str = \"\"\n    author: str | None = None\n    published_at: str | None = None\n    score: float = 0.0\n\n\nclass ResearchFact(StudioModel):\n    fact_id: str\n    claim: str\n    source_ids: list[str] = Field(default_factory=list)\n    confidence: float = Field(default=0.5, ge=0.0, le=1.0)\n    numeric_values: list[str] = Field(default_factory=list)\n    visual_hint: str = \"\"\n\n\nclass ResearchBundle(StudioModel):\n    topic: str\n    summary: str\n    facts: list[ResearchFact]\n    numbers: list[ResearchFact] = Field(default_factory=list)\n    references: list[SourceRecord]\n    comparisons: list[str] = Field(default_factory=list)\n    visual_ideas: list[str] = Field(default_factory=list)\n    limitations: list[str] = Field(default_factory=list)\n    research_hash: str = \"\"\n\n\nclass ValidationIssue(StudioModel):\n    severity: str\n    code: str\n    message: str\n    claim: str | None = None\n    source_ids: list[str] = Field(default_factory=list)\n\n\nclass ValidationReport(StudioModel):\n    passed: bool\n    coverage_score: float = Field(ge=0.0, le=1.0)\n    source_diversity: int = 0\n    issues: list[ValidationIssue] = Field(default_factory=list)\n    validated_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())\n\n\n# --------------------------------------------------------------------------- #\n# Script\n# --------------------------------------------------------------------------- #\nclass ScriptBeat(StudioModel):\n    beat_id: str\n    purpose: str\n    narration: str\n    evidence_refs: list[str] = Field(default_factory=list)\n    words: int = 0\n    start_s: float = 0.0\n    end_s: float = 0.0\n    estimated_duration_s: float = 0.0\n\n\nclass ScriptPackage(StudioModel):\n    topic: str\n    hook: str\n    curiosity: str\n    scientific_explanation: str\n    escalation: str\n    ending: str\n    beats: list[ScriptBeat]\n    estimated_duration_s: float = 0.0\n    total_words: int = 0\n    tone: str = \"curious, precise, escalating\"\n    script_hash: str = \"\"\n\n\n# --------------------------------------------------------------------------- #\n# Storyboard + scene graph\n# --------------------------------------------------------------------------- #\nclass AssetRequirement(StudioModel):\n    asset_id: str\n    asset_type: str\n    label: str = \"\"\n    variant: str = \"default\"\n    palette_role: str = \"primary\"\n    source_prompt: str = \"\"\n    tags: list[str] = Field(default_factory=list)\n\n\nclass SceneObject(StudioModel):\n    \"\"\"Placement of one asset on the canvas. Coordinates are normalised 0..1.\"\"\"\n\n    asset_id: str\n    x: float = Field(ge=0.0, le=1.0)\n    y: float = Field(ge=0.0, le=1.0)\n    width: float = Field(gt=0.0, le=1.0)\n    height: float = Field(gt=0.0, le=1.0)\n    rotation: float = 0.0\n    z_index: int = 0\n    opacity: float = Field(default=1.0, ge=0.0, le=1.0)\n\n\nclass CameraPlan(StudioModel):\n    shot: str = \"wide\"\n    zoom_start: float = 1.0\n    zoom_end: float = 1.04\n    pan_x: float = 0.0\n    pan_y: float = 0.0\n    framing: str = \"center\"\n\n\nclass AnimationPlaceholder(StudioModel):\n    \"\"\"Phase-2 animation intent. Carried through the storyboard, not rendered here.\"\"\"\n\n    target_asset_id: str\n    kind: str\n    start_s: float = 0.0\n    duration_s: float = 1.0\n    easing: str = \"easeInOut\"\n\n\nclass DashboardSpec(StudioModel):\n    experiment_id: str = \"EXPERIMENT #001\"\n    status_label: str = \"Simulation Status\"\n    status_value: str = \"Running\"\n    metric_label: str = \"VALUE\"\n    metric_value: str = \"\u2014\"\n    severity: str = \"normal\"\n\n\nclass Scene(StudioModel):\n    scene_id: str\n    duration_s: float = Field(gt=0.0)\n    narration: str\n    title: str = \"\"\n    objects: list[SceneObject] = Field(default_factory=list)\n    camera: CameraPlan = Field(default_factory=CameraPlan)\n    animations: list[AnimationPlaceholder] = Field(default_factory=list)\n    dashboard: DashboardSpec = Field(default_factory=DashboardSpec)\n    text: list[str] = Field(default_factory=list)\n    icons: list[str] = Field(default_factory=list)\n    background: str = \"paper\"\n    motion: str = \"hold\"\n    transition: str = \"cut\"\n    asset_requirements: list[AssetRequirement] = Field(default_factory=list)\n\n\nclass Storyboard(StudioModel):\n    topic: str\n    canvas_width: int = 1080\n    canvas_height: int = 1920\n    style_name: str = \"scientific-simulation-flat\"\n    scenes: list[Scene]\n    asset_catalog: list[AssetRequirement] = Field(default_factory=list)\n    estimated_duration_s: float = 0.0\n    storyboard_hash: str = \"\"\n\n\n# --------------------------------------------------------------------------- #\n# Phase 2 \u2014 audio, captions, render, SEO\n# --------------------------------------------------------------------------- #\nclass AudioClip(StudioModel):\n    scene_id: str\n    text: str\n    path: str\n    start_s: float\n    end_s: float\n    duration_s: float\n    provider: str\n    audio_hash: str\n\n\nclass AudioTimeline(StudioModel):\n    clips: list[AudioClip]\n    voice_track: str\n    duration_s: float\n    timeline_hash: str\n\n\nclass CaptionSegment(StudioModel):\n    index: int\n    text: str\n    start_s: float\n    end_s: float\n    scene_id: str\n\n\nclass SEOPackage(StudioModel):\n    title: str\n    description: str\n    hashtags: list[str] = Field(default_factory=list)\n    tags: list[str] = Field(default_factory=list)\n    filename: str = \"short.mp4\"\n    seo_score: int = 0\n\n\nclass RenderReport(StudioModel):\n    video_path: str\n    width: int\n    height: int\n    fps: float\n    duration_s: float\n    video_codec: str\n    audio_codec: str | None = None\n    has_audio: bool = False\n    file_size_bytes: int = 0\n    render_hash: str = \"\"\n    passed: bool = False\n    checks: list[str] = Field(default_factory=list)\n\n\n# --------------------------------------------------------------------------- #\n# Cache + run bookkeeping\n# --------------------------------------------------------------------------- #\nclass AssetMetadata(StudioModel):\n    asset_hash: str\n    prompt_hash: str\n    svg_hash: str\n    asset_id: str\n    asset_type: str\n    source_prompt: str\n    topic: str\n    created_at: str\n    updated_at: str\n    reuse_counter: int = 0\n    embedding: list[float] | None = None\n    generator_version: str = \"svg-template-v1\"\n\n\nclass RunManifest(StudioModel):\n    run_id: str\n    topic: str\n    mode: str\n    created_at: str\n    project_root: str\n    run_directory: str\n    hardware: dict[str, Any]\n    research_hash: str\n    script_hash: str\n    storyboard_hash: str\n    asset_hashes: dict[str, str]\n    files: dict[str, str]\n    validation_passed: bool = False\n    estimated_duration_s: float = 0.0\n    warnings: list[str] = Field(default_factory=list)\n    video_hash: str | None = None\n    render_report: dict[str, Any] | None = None\n",
    "config.py": "from __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nimport yaml\nfrom pydantic import BaseModel, ConfigDict, Field\n\n\nclass ConfigModel(BaseModel):\n    model_config = ConfigDict(extra=\"ignore\")\n\n\nclass ProjectConfig(ConfigModel):\n    name: str = \"autonomous-scientific-animation-studio\"\n    root: str = \"\"\n    random_seed: int = 42\n\n\nclass LLMConfig(ConfigModel):\n    # Priority 1: local Qwen. Priority 2: OpenAI, used only if local fails.\n    gpu_model: str = \"Qwen/Qwen2.5-7B-Instruct\"\n    cpu_model: str = \"Qwen/Qwen2.5-1.5B-Instruct\"\n    openai_model: str = \"gpt-4.1-mini\"\n    use_openai_fallback: bool = True\n    prefer_local: bool = True\n    quantization: str = \"4bit\"  # 4bit | none\n    temperature: float = 0.15\n    max_new_tokens: int = 2400\n    context_tokens: int = 7000\n    unload_after_pipeline: bool = True\n\n\nclass SearchConfig(ConfigModel):\n    providers: list[str] = Field(default_factory=lambda: [\"ddgs\", \"wikipedia\", \"crossref\", \"openalex\", \"arxiv\"])\n    max_results_per_provider: int = 5\n    timeout_seconds: int = 20\n    min_sources: int = 5\n    rss_feeds: list[str] = Field(default_factory=lambda: [\n        \"https://www.nasa.gov/rss/dyn/breaking_news.rss\",\n        \"https://www.sciencedaily.com/rss/top/science.xml\",\n        \"https://phys.org/rss-feed/\",\n    ])\n\n\nclass ResearchConfig(ConfigModel):\n    minimum_supported_facts: int = 5\n    minimum_non_wikipedia_sources: int = 2\n    strict_numeric_validation: bool = True\n\n\nclass ScriptConfig(ConfigModel):\n    target_duration_seconds: float = 55.0\n    target_words_min: int = 105\n    target_words_max: int = 155\n    words_per_second: float = 2.55\n    scene_count_min: int = 7\n    scene_count_max: int = 10\n\n\nclass CanvasConfig(ConfigModel):\n    width: int = 1080\n    height: int = 1920\n    safe_margin: int = 64\n\n\nclass StyleConfig(ConfigModel):\n    name: str = \"scientific-simulation-flat\"\n    palette: dict[str, str] = Field(default_factory=lambda: {\n        \"paper\": \"#F7F7F4\", \"ink\": \"#17202A\", \"primary\": \"#2D9CDB\",\n        \"secondary\": \"#566573\", \"dark\": \"#202A33\", \"warning\": \"#E74C3C\",\n        \"sun\": \"#F5C542\", \"white\": \"#FFFFFF\",\n    })\n    stroke_width: float = 5.0\n    font_family: str = \"DejaVu Sans\"\n    title_weight: int = 800\n    body_weight: int = 500\n    no_gradients: bool = True\n    no_textures: bool = True\n    no_raster: bool = True\n\n\nclass CacheConfig(ConfigModel):\n    enabled: bool = True\n    reuse_assets: bool = True\n    reuse_structured_outputs: bool = True\n    optional_embeddings: bool = False\n\n\nclass SVGConfig(ConfigModel):\n    generator_version: str = \"svg-template-v1\"\n    asset_viewbox: int = 1000\n    max_scene_objects: int = 12\n    validate_xml: bool = True\n\n\nclass PreviewConfig(ConfigModel):\n    # Static storyboard preview settings (no animation yet \u2014 Phase 2).\n    rasterize: bool = True  # CairoSVG -> PNG contact sheet for reliable inline display\n    contact_columns: int = 3\n    thumb_width: int = 360\n    thumb_height: int = 640\n\n\nclass VoiceConfig(ConfigModel):\n    # Phase 2. Edge TTS (free, online) primary; espeak-ng offline fallback.\n    provider: str = \"edge\"\n    voice: str = \"en-US-GuyNeural\"\n    rate: str = \"+8%\"\n    pitch: str = \"+0Hz\"\n    local_voice: str = \"en-us\"\n    local_words_per_minute: int = 175\n    scene_tail_silence: float = 0.18\n    sample_rate: int = 48000\n\n\nclass CaptionConfig(ConfigModel):\n    enabled: bool = True\n    font_name: str = \"DejaVu Sans\"\n    font_size: int = 56\n    max_words: int = 4\n    max_chars: int = 25\n    margin_left: int = 70\n    margin_right: int = 70\n    margin_vertical: int = 285\n    primary_color: str = \"&H00FFFFFF\"\n    outline_color: str = \"&H00101010\"\n    back_color: str = \"&H98000000\"\n\n\nclass AudioConfig(ConfigModel):\n    background_enabled: bool = True\n    music_volume: float = 0.10\n    sfx_enabled: bool = True\n    sfx_volume: float = 0.22\n    voice_loudness_lufs: float = -16.0\n\n\nclass RenderConfig(ConfigModel):\n    width: int = 1080\n    height: int = 1920\n    fps: int = 30\n    crf: int = 19\n    preset: str = \"veryfast\"\n    pixel_format: str = \"yuv420p\"\n    audio_bitrate: str = \"192k\"\n    scene_fade_seconds: float = 0.06\n    cleanup_temporary_files: bool = False\n    validate_output: bool = True\n\n\nclass SEOConfig(ConfigModel):\n    enabled: bool = True\n    max_title_chars: int = 70\n    hashtag_count: int = 5\n    tag_count: int = 16\n\n\nclass StudioConfig(ConfigModel):\n    project: ProjectConfig = Field(default_factory=ProjectConfig)\n    llm: LLMConfig = Field(default_factory=LLMConfig)\n    search: SearchConfig = Field(default_factory=SearchConfig)\n    research: ResearchConfig = Field(default_factory=ResearchConfig)\n    script: ScriptConfig = Field(default_factory=ScriptConfig)\n    canvas: CanvasConfig = Field(default_factory=CanvasConfig)\n    style: StyleConfig = Field(default_factory=StyleConfig)\n    cache: CacheConfig = Field(default_factory=CacheConfig)\n    svg: SVGConfig = Field(default_factory=SVGConfig)\n    preview: PreviewConfig = Field(default_factory=PreviewConfig)\n    voice: VoiceConfig = Field(default_factory=VoiceConfig)\n    captions: CaptionConfig = Field(default_factory=CaptionConfig)\n    audio: AudioConfig = Field(default_factory=AudioConfig)\n    render: RenderConfig = Field(default_factory=RenderConfig)\n    seo: SEOConfig = Field(default_factory=SEOConfig)\n\n\nDEFAULT_CONFIG = StudioConfig().model_dump(mode=\"json\")\n\n\ndef ensure_default_config(path: Path) -> Path:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not path.exists():\n        path.write_text(yaml.safe_dump(DEFAULT_CONFIG, sort_keys=False), encoding=\"utf-8\")\n    return path\n\n\ndef load_config(path: Path | None = None) -> StudioConfig:\n    root = Path(os.environ.get(\"AUTOSTUDIO_ROOT\", \".\")).resolve()\n    config_path = path or root / \"config\" / \"config.yaml\"\n    ensure_default_config(config_path)\n    data = yaml.safe_load(config_path.read_text(encoding=\"utf-8\")) or {}\n    data.setdefault(\"project\", {})[\"root\"] = str(root)\n    return StudioConfig.model_validate(data)\n",
    "logging_utils.py": "from __future__ import annotations\n\nimport logging\nimport os\nfrom pathlib import Path\n\n\ndef configure_logging(name: str = \"autostudio\") -> logging.Logger:\n    \"\"\"Idempotent logger factory. Streams to stdout and appends to logs/studio.log.\"\"\"\n    root = Path(os.environ.get(\"AUTOSTUDIO_ROOT\", \".\")).resolve()\n    log_dir = root / \"logs\"\n    log_dir.mkdir(parents=True, exist_ok=True)\n    logger = logging.getLogger(name)\n    if logger.handlers:\n        return logger\n    logger.setLevel(logging.INFO)\n    formatter = logging.Formatter(\"%(asctime)s | %(levelname)s | %(name)s | %(message)s\")\n    stream = logging.StreamHandler()\n    stream.setFormatter(formatter)\n    file_handler = logging.FileHandler(log_dir / \"studio.log\", encoding=\"utf-8\")\n    file_handler.setFormatter(formatter)\n    logger.addHandler(stream)\n    logger.addHandler(file_handler)\n    logger.propagate = False\n    return logger\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

In [ ]:
from autostudio.config import ensure_default_config, load_config

CONFIG_PATH = ensure_default_config(PROJECT_ROOT / "config" / "config.yaml")
config = load_config(CONFIG_PATH)
print("Config written to:", CONFIG_PATH)
print("Canvas:", config.canvas.width, "x", config.canvas.height, "| render fps:", config.render.fps)
print("Voice:", config.voice.voice, "| loudness target:", config.audio.voice_loudness_lufs, "LUFS")
print("Primary LLM:", config.llm.gpu_model)

## 4. Model loading and hardware detection

`hardware.py` auto-detects the accelerator (T4 / L4 / A100 / CPU) and dtype
(bf16 on Ampere+, fp16 on older GPUs, fp32 on CPU). `llm.py` is a local-first
JSON LLM client: lazy 4-bit Qwen, optional OpenAI fallback, content-cached
responses, and robust JSON extraction + one repair pass.

In [ ]:
# Section 4 — hardware detection + local-first LLM
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "hardware.py": "from __future__ import annotations\n\nimport gc\nimport os\nimport random\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass HardwareProfile:\n    accelerator: str\n    device_name: str\n    total_vram_gb: float\n    compute_capability: str\n    supports_bf16: bool\n    recommended_dtype: str\n    recommended_quantization: str\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\ndef detect_hardware() -> HardwareProfile:\n    \"\"\"Auto-detect T4/L4/A100/CPU. Chooses bf16 on Ampere+, else fp16, else fp32.\"\"\"\n    try:\n        import torch\n    except Exception:\n        return HardwareProfile(\"cpu\", \"CPU\", 0.0, \"n/a\", False, \"float32\", \"none\")\n    if not torch.cuda.is_available():\n        return HardwareProfile(\"cpu\", \"CPU\", 0.0, \"n/a\", False, \"float32\", \"none\")\n    index = torch.cuda.current_device()\n    props = torch.cuda.get_device_properties(index)\n    major, minor = torch.cuda.get_device_capability(index)\n    total = props.total_memory / (1024 ** 3)\n    bf16 = bool(torch.cuda.is_bf16_supported())\n    return HardwareProfile(\n        \"cuda\", props.name, round(total, 2), f\"{major}.{minor}\", bf16,\n        \"bfloat16\" if bf16 else \"float16\", \"4bit\",\n    )\n\n\ndef seed_everything(seed: int) -> None:\n    os.environ[\"PYTHONHASHSEED\"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n    except Exception:\n        pass\n\n\ndef release_gpu_memory() -> None:\n    gc.collect()\n    try:\n        import torch\n\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n            torch.cuda.ipc_collect()\n    except Exception:\n        pass\n",
    "llm.py": "from __future__ import annotations\n\nimport json\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\nfrom tenacity import retry, stop_after_attempt, wait_exponential\n\nfrom .config import StudioConfig\nfrom .hardware import detect_hardware, release_gpu_memory, seed_everything\nfrom .hashing import atomic_write_json, hash_value, read_json\nfrom .logging_utils import configure_logging\n\n\nclass LLMError(RuntimeError):\n    pass\n\n\nclass LLMClient:\n    \"\"\"Local-first JSON LLM. Qwen (HF) is primary; OpenAI is an optional fallback.\n\n    Every response is content-addressed and cached, so re-runs never re-generate\n    identical prompts. The model is loaded lazily on first real call.\n    \"\"\"\n\n    def __init__(self, config: StudioConfig, cache_dir: Path):\n        self.config = config\n        self.cache_dir = cache_dir\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n        self.hardware = detect_hardware()\n        self.logger = configure_logging(\"autostudio.llm\")\n        self._tokenizer = None\n        self._model = None\n        seed_everything(config.project.random_seed)\n\n    @property\n    def openai_available(self) -> bool:\n        return bool(os.getenv(\"OPENAI_API_KEY\", \"\").strip())\n\n    def _extract_json(self, text: str) -> dict[str, Any]:\n        \"\"\"Robust JSON recovery: strips fences, then brace-matches the first object.\"\"\"\n        cleaned = text.strip().replace(\"```json\", \"\").replace(\"```\", \"\")\n        try:\n            return json.loads(cleaned)\n        except json.JSONDecodeError:\n            pass\n        for start in [m.start() for m in re.finditer(r\"\\{\", cleaned)]:\n            depth = 0\n            in_string = False\n            escaped = False\n            for index in range(start, len(cleaned)):\n                char = cleaned[index]\n                if escaped:\n                    escaped = False\n                    continue\n                if char == \"\\\\\":\n                    escaped = True\n                    continue\n                if char == '\"':\n                    in_string = not in_string\n                if in_string:\n                    continue\n                if char == \"{\":\n                    depth += 1\n                elif char == \"}\":\n                    depth -= 1\n                    if depth == 0:\n                        try:\n                            return json.loads(cleaned[start:index + 1])\n                        except json.JSONDecodeError:\n                            break\n        raise LLMError(\"Model output did not contain valid JSON.\")\n\n    def _load_local(self) -> None:\n        if self._model is not None:\n            return\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n        model_id = self.config.llm.gpu_model if self.hardware.accelerator == \"cuda\" else self.config.llm.cpu_model\n        self.logger.info(\"Loading %s on %s\", model_id, self.hardware.device_name)\n        self._tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)\n        kwargs: dict[str, Any] = {\"low_cpu_mem_usage\": True, \"trust_remote_code\": False}\n        if self.hardware.accelerator == \"cuda\":\n            dtype = torch.bfloat16 if self.hardware.supports_bf16 else torch.float16\n            if self.config.llm.quantization == \"4bit\":\n                kwargs[\"quantization_config\"] = BitsAndBytesConfig(\n                    load_in_4bit=True,\n                    bnb_4bit_quant_type=\"nf4\",\n                    bnb_4bit_use_double_quant=True,\n                    bnb_4bit_compute_dtype=dtype,\n                )\n            kwargs[\"device_map\"] = \"auto\"\n            kwargs[\"dtype\"] = dtype\n        else:\n            kwargs.update({\"device_map\": {\"\": \"cpu\"}, \"dtype\": torch.float32})\n        try:\n            self._model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)\n        except TypeError:\n            # Older transformers use torch_dtype instead of dtype.\n            dtype = kwargs.pop(\"dtype\", None)\n            if dtype is not None:\n                kwargs[\"torch_dtype\"] = dtype\n            self._model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)\n        self._model.eval()\n\n    @retry(stop=stop_after_attempt(2), wait=wait_exponential(multiplier=1, min=1, max=4), reraise=True)\n    def _generate_local(self, system_prompt: str, user_prompt: str, max_new_tokens: int) -> dict[str, Any]:\n        import torch\n\n        self._load_local()\n        messages = [{\"role\": \"system\", \"content\": system_prompt}, {\"role\": \"user\", \"content\": user_prompt}]\n        prompt = self._tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        inputs = self._tokenizer(prompt, return_tensors=\"pt\", truncation=True, max_length=self.config.llm.context_tokens)\n        device = self._model.get_input_embeddings().weight.device\n        inputs = {key: value.to(device) for key, value in inputs.items()}\n        deterministic = self.config.llm.temperature <= 0.2\n        with torch.inference_mode():\n            output = self._model.generate(\n                **inputs,\n                max_new_tokens=max_new_tokens,\n                do_sample=not deterministic,\n                temperature=max(self.config.llm.temperature, 0.01),\n                top_p=0.9,\n                repetition_penalty=1.05,\n                pad_token_id=self._tokenizer.eos_token_id,\n            )\n        generated = output[0, inputs[\"input_ids\"].shape[1]:]\n        text = self._tokenizer.decode(generated, skip_special_tokens=True)\n        try:\n            return self._extract_json(text)\n        except LLMError:\n            # One deterministic repair pass to coerce malformed output into JSON.\n            repair = f\"Repair into one valid JSON object. Return JSON only:\\n\\n{text}\"\n            repair_messages = [{\"role\": \"system\", \"content\": \"Repair malformed JSON.\"}, {\"role\": \"user\", \"content\": repair}]\n            repair_text = self._tokenizer.apply_chat_template(repair_messages, tokenize=False, add_generation_prompt=True)\n            repair_inputs = self._tokenizer(repair_text, return_tensors=\"pt\", truncation=True, max_length=5000)\n            repair_inputs = {key: value.to(device) for key, value in repair_inputs.items()}\n            with torch.inference_mode():\n                repaired = self._model.generate(\n                    **repair_inputs, max_new_tokens=min(max_new_tokens, 1800),\n                    do_sample=False, pad_token_id=self._tokenizer.eos_token_id,\n                )\n            return self._extract_json(self._tokenizer.decode(repaired[0, repair_inputs[\"input_ids\"].shape[1]:], skip_special_tokens=True))\n\n    def _generate_openai(self, system_prompt: str, user_prompt: str, max_new_tokens: int) -> dict[str, Any]:\n        if not self.openai_available:\n            raise LLMError(\"OPENAI_API_KEY is unavailable.\")\n        from openai import OpenAI\n\n        client = OpenAI(api_key=os.getenv(\"OPENAI_API_KEY\"))\n        response = client.chat.completions.create(\n            model=self.config.llm.openai_model,\n            messages=[{\"role\": \"system\", \"content\": system_prompt}, {\"role\": \"user\", \"content\": user_prompt}],\n            temperature=self.config.llm.temperature,\n            max_tokens=max_new_tokens,\n            response_format={\"type\": \"json_object\"},\n        )\n        return json.loads(response.choices[0].message.content)\n\n    def generate_json(\n        self,\n        system_prompt: str,\n        user_prompt: str,\n        *,\n        cache_namespace: str,\n        max_new_tokens: int | None = None,\n        force_refresh: bool = False,\n    ) -> dict[str, Any]:\n        limit = max_new_tokens or self.config.llm.max_new_tokens\n        key = hash_value({\n            \"model\": self.config.llm.gpu_model, \"system\": system_prompt, \"user\": user_prompt,\n            \"limit\": limit, \"temperature\": self.config.llm.temperature,\n        })\n        path = self.cache_dir / cache_namespace / f\"{key}.json\"\n        if path.exists() and not force_refresh:\n            cached = read_json(path)\n            if isinstance(cached, dict):\n                return cached\n        local_error = None\n        if self.config.llm.prefer_local:\n            try:\n                result = self._generate_local(system_prompt, user_prompt, limit)\n                atomic_write_json(path, result)\n                return result\n            except Exception as exc:\n                local_error = exc\n                self.logger.warning(\"Local LLM failed: %s\", exc)\n                if \"out of memory\" in str(exc).lower():\n                    release_gpu_memory()\n        if self.config.llm.use_openai_fallback and self.openai_available:\n            result = self._generate_openai(system_prompt, user_prompt, limit)\n            atomic_write_json(path, result)\n            return result\n        raise LLMError(f\"All LLM paths failed. Local error: {local_error}\")\n\n    def unload(self) -> None:\n        self._model = None\n        self._tokenizer = None\n        release_gpu_memory()\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

In [ ]:
from autostudio.hardware import detect_hardware

hardware = detect_hardware()
print("Accelerator:", hardware.accelerator, "|", hardware.device_name)
print("VRAM (GB):", hardware.total_vram_gb, "| dtype:", hardware.recommended_dtype)
print("OpenAI fallback available:", bool(os.getenv("OPENAI_API_KEY", "").strip()))

## 5. Search layer

Free sources only — no paid APIs. DDGS primary; Wikipedia, Crossref, OpenAlex, arXiv add scientific depth and diversity; science RSS feeds power trending-topic discovery. Deduplicated, scored, and cached per topic.

In [ ]:
# Section 5 — free multi-provider search
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "search.py": "from __future__ import annotations\n\nimport html\nimport re\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport feedparser\nimport requests\nfrom tenacity import retry, stop_after_attempt, wait_exponential\n\nfrom .config import StudioConfig\nfrom .hashing import atomic_write_json, hash_value, read_json\nfrom .logging_utils import configure_logging\nfrom .schemas import SourceRecord\n\n\nclass SearchService:\n    \"\"\"Free-only web search. No paid APIs (no Tavily/SerpAPI/Google). Cached per topic.\"\"\"\n\n    def __init__(self, config: StudioConfig, cache_dir: Path):\n        self.config = config\n        self.cache_dir = cache_dir / \"search\"\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n        self.session = requests.Session()\n        self.session.headers.update({\"User-Agent\": \"AutonomousScientificAnimationStudio/0.2\"})\n        self.logger = configure_logging(\"autostudio.search\")\n\n    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=1, max=5), reraise=True)\n    def _get_json(self, url: str, params: dict | None = None) -> dict:\n        response = self.session.get(url, params=params, timeout=self.config.search.timeout_seconds)\n        response.raise_for_status()\n        return response.json()\n\n    def _source_id(self, provider: str, title: str, url: str | None) -> str:\n        return f\"{provider[:3].lower()}-{hash_value({'title': title, 'url': url})[:10]}\"\n\n    def _deduplicate(self, sources: Iterable[SourceRecord]) -> list[SourceRecord]:\n        output: list[SourceRecord] = []\n        seen: set[str] = set()\n        for source in sources:\n            key = re.sub(r\"\\W+\", \"\", source.title.lower())[:120]\n            if not key or key in seen:\n                continue\n            seen.add(key)\n            output.append(source)\n        return output\n\n    def search_ddgs(self, query: str, limit: int) -> list[SourceRecord]:\n        try:\n            from ddgs import DDGS\n\n            output = []\n            for item in DDGS().text(query, max_results=limit):\n                title = str(item.get(\"title\") or \"\").strip()\n                url = item.get(\"href\") or item.get(\"url\")\n                if title:\n                    output.append(SourceRecord(\n                        source_id=self._source_id(\"ddgs\", title, url), provider=\"DDGS\",\n                        title=title, url=url, snippet=str(item.get(\"body\") or \"\"), score=0.72,\n                    ))\n            return output\n        except Exception as exc:\n            self.logger.warning(\"DDGS failed: %s\", exc)\n            return []\n\n    def search_wikipedia(self, query: str, limit: int) -> list[SourceRecord]:\n        try:\n            data = self._get_json(\"https://en.wikipedia.org/w/api.php\", {\n                \"action\": \"query\", \"list\": \"search\", \"srsearch\": query,\n                \"format\": \"json\", \"utf8\": 1, \"srlimit\": limit,\n            })\n            output = []\n            for item in data.get(\"query\", {}).get(\"search\", []):\n                title = item.get(\"title\", \"\")\n                url = \"https://en.wikipedia.org/wiki/\" + title.replace(\" \", \"_\")\n                snippet = re.sub(\"<[^>]+>\", \"\", html.unescape(item.get(\"snippet\", \"\")))\n                output.append(SourceRecord(\n                    source_id=self._source_id(\"wikipedia\", title, url), provider=\"Wikipedia\",\n                    title=title, url=url, snippet=snippet, score=0.58,\n                ))\n            return output\n        except Exception as exc:\n            self.logger.warning(\"Wikipedia failed: %s\", exc)\n            return []\n\n    def search_crossref(self, query: str, limit: int) -> list[SourceRecord]:\n        try:\n            data = self._get_json(\"https://api.crossref.org/works\", {\n                \"query.title\": query, \"rows\": limit,\n                \"select\": \"DOI,title,URL,author,published-online,published-print,abstract\",\n            })\n            output = []\n            for item in data.get(\"message\", {}).get(\"items\", []):\n                title = \" \".join(item.get(\"title\") or []).strip()\n                if not title:\n                    continue\n                url = item.get(\"URL\") or (f\"https://doi.org/{item['DOI']}\" if item.get(\"DOI\") else None)\n                authors = item.get(\"author\") or []\n                author = \", \".join(\" \".join(filter(None, [a.get(\"given\"), a.get(\"family\")])) for a in authors[:3]) or None\n                published = None\n                for key in (\"published-print\", \"published-online\"):\n                    parts = ((item.get(key) or {}).get(\"date-parts\") or [])\n                    if parts:\n                        published = \"-\".join(str(value) for value in parts[0])\n                        break\n                output.append(SourceRecord(\n                    source_id=self._source_id(\"crossref\", title, url), provider=\"Crossref\",\n                    title=title, url=url, snippet=re.sub(\"<[^>]+>\", \"\", item.get(\"abstract\") or \"\")[:1000],\n                    author=author, published_at=published, score=0.90,\n                ))\n            return output\n        except Exception as exc:\n            self.logger.warning(\"Crossref failed: %s\", exc)\n            return []\n\n    def search_openalex(self, query: str, limit: int) -> list[SourceRecord]:\n        try:\n            data = self._get_json(\"https://api.openalex.org/works\", {\n                \"search\": query, \"per-page\": limit,\n                \"select\": \"id,title,doi,publication_year,primary_location,authorships\",\n            })\n            output = []\n            for item in data.get(\"results\", []):\n                title = item.get(\"title\") or \"\"\n                location = item.get(\"primary_location\") or {}\n                url = location.get(\"landing_page_url\") or item.get(\"doi\") or item.get(\"id\")\n                authors = \", \".join(str((a.get(\"author\") or {}).get(\"display_name\") or \"\") for a in (item.get(\"authorships\") or [])[:3]).strip(\", \") or None\n                output.append(SourceRecord(\n                    source_id=self._source_id(\"openalex\", title, url), provider=\"OpenAlex\",\n                    title=title, url=url, author=authors,\n                    published_at=str(item.get(\"publication_year\") or \"\") or None, score=0.88,\n                ))\n            return output\n        except Exception as exc:\n            self.logger.warning(\"OpenAlex failed: %s\", exc)\n            return []\n\n    def search_arxiv(self, query: str, limit: int) -> list[SourceRecord]:\n        try:\n            response = self.session.get(\"https://export.arxiv.org/api/query\", params={\n                \"search_query\": f\"all:{query}\", \"start\": 0, \"max_results\": limit,\n            }, timeout=self.config.search.timeout_seconds)\n            response.raise_for_status()\n            feed = feedparser.loads(response.text)\n            output = []\n            for entry in feed.entries:\n                title = re.sub(r\"\\s+\", \" \", entry.get(\"title\", \"\")).strip()\n                url = entry.get(\"link\")\n                output.append(SourceRecord(\n                    source_id=self._source_id(\"arxiv\", title, url), provider=\"arXiv\",\n                    title=title, url=url,\n                    snippet=re.sub(r\"\\s+\", \" \", entry.get(\"summary\", \"\")).strip()[:1200],\n                    author=\", \".join(a.get(\"name\", \"\") for a in entry.get(\"authors\", [])[:3]) or None,\n                    published_at=entry.get(\"published\"), score=0.92,\n                ))\n            return output\n        except Exception as exc:\n            self.logger.warning(\"arXiv failed: %s\", exc)\n            return []\n\n    def search_topic(self, topic: str, force_refresh: bool = False) -> list[SourceRecord]:\n        key = hash_value({\n            \"topic\": topic, \"providers\": self.config.search.providers,\n            \"limit\": self.config.search.max_results_per_provider,\n        })\n        path = self.cache_dir / f\"{key}.json\"\n        if path.exists() and not force_refresh:\n            return [SourceRecord.model_validate(item) for item in read_json(path, [])]\n        results: list[SourceRecord] = []\n        for provider in self.config.search.providers:\n            method = getattr(self, f\"search_{provider}\", None)\n            if callable(method):\n                results.extend(method(topic, self.config.search.max_results_per_provider))\n        results = sorted(self._deduplicate(results), key=lambda item: item.score, reverse=True)\n        atomic_write_json(path, [item.model_dump(mode=\"json\") for item in results])\n        return results\n\n    def discover_trending_titles(self, limit: int = 24) -> list[str]:\n        \"\"\"Free trend discovery via science RSS feeds + DDGS. Used by auto topic mode.\"\"\"\n        titles: list[str] = []\n        for feed_url in self.config.search.rss_feeds:\n            try:\n                feed = feedparser.parse(feed_url)\n                titles.extend(str(entry.get(\"title\") or \"\").strip() for entry in feed.entries[:10])\n            except Exception as exc:\n                self.logger.warning(\"RSS failed %s: %s\", feed_url, exc)\n        titles.extend(item.title for item in self.search_ddgs(\"latest science technology discovery research\", 10))\n        output, seen = [], set()\n        for title in titles:\n            key = re.sub(r\"\\W+\", \"\", title.lower())\n            if title and key not in seen:\n                seen.add(key)\n                output.append(title)\n        return output[:limit]\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 6. Research layer and scientific validation

LLM synthesis grounded strictly in retrieved sources (summary, source-backed facts, numbers, comparisons, visual ideas, limitations), then a fully deterministic validator (source references, cited numerics, source diversity). Output is structured JSON.

In [ ]:
# Section 6 — research synthesis + deterministic validation
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "research.py": "from __future__ import annotations\n\nimport json\nimport re\nfrom pathlib import Path\n\nfrom .config import StudioConfig\nfrom .hashing import atomic_write_json, hash_value, read_json\nfrom .llm import LLMClient\nfrom .logging_utils import configure_logging\nfrom .schemas import ResearchBundle, ResearchFact, SourceRecord, ValidationIssue, ValidationReport\n\n\nclass ScientificValidator:\n    \"\"\"Deterministic checks. No LLM: every rule is inspectable and reproducible.\"\"\"\n\n    def __init__(self, config: StudioConfig):\n        self.config = config\n\n    def validate(self, bundle: ResearchBundle) -> ValidationReport:\n        issues: list[ValidationIssue] = []\n        valid_ids = {source.source_id for source in bundle.references}\n        supported = 0\n        for fact in bundle.facts + bundle.numbers:\n            missing = [source_id for source_id in fact.source_ids if source_id not in valid_ids]\n            if missing:\n                issues.append(ValidationIssue(\n                    severity=\"error\", code=\"missing_source_reference\",\n                    message=f\"Unknown source IDs: {missing}\", claim=fact.claim, source_ids=fact.source_ids,\n                ))\n            if fact.source_ids and not missing:\n                supported += 1\n            else:\n                issues.append(ValidationIssue(\n                    severity=\"warning\", code=\"unsupported_claim\",\n                    message=\"Claim has no valid source reference.\", claim=fact.claim, source_ids=fact.source_ids,\n                ))\n            contains_number = bool(re.search(r\"\\b\\d+(?:\\.\\d+)?\\s*(?:%|km|m/s|kg|years?|hours?|\u00b0C|K|Hz|W|J)?\\b\", fact.claim))\n            if contains_number and self.config.research.strict_numeric_validation and not fact.source_ids:\n                issues.append(ValidationIssue(\n                    severity=\"error\", code=\"uncited_number\",\n                    message=\"Numeric claim requires a source.\", claim=fact.claim,\n                ))\n        total = max(1, len(bundle.facts) + len(bundle.numbers))\n        coverage = supported / total\n        non_wikipedia = sum(1 for source in bundle.references if source.provider.lower() != \"wikipedia\")\n        if non_wikipedia < self.config.research.minimum_non_wikipedia_sources:\n            issues.append(ValidationIssue(\n                severity=\"error\", code=\"source_diversity_low\",\n                message=f\"Only {non_wikipedia} non-Wikipedia sources are available.\",\n            ))\n        if len(bundle.facts) < self.config.research.minimum_supported_facts:\n            issues.append(ValidationIssue(\n                severity=\"warning\", code=\"too_few_facts\",\n                message=f\"Research contains only {len(bundle.facts)} facts.\",\n            ))\n        return ValidationReport(\n            passed=not any(issue.severity == \"error\" for issue in issues) and coverage >= 0.65,\n            coverage_score=round(coverage, 3),\n            source_diversity=len({source.provider for source in bundle.references}),\n            issues=issues,\n        )\n\n\nclass ResearchService:\n    \"\"\"LLM synthesis grounded in supplied sources, then deterministic validation.\"\"\"\n\n    SYSTEM_PROMPT = (\n        \"You are a scientific research editor. Use only supplied source records. \"\n        \"Never invent citations. Separate established facts from uncertainty. \"\n        \"Numeric statements must point to source IDs. Return valid JSON only.\"\n    )\n\n    def __init__(self, config: StudioConfig, llm: LLMClient, cache_dir: Path):\n        self.config = config\n        self.llm = llm\n        self.cache_dir = cache_dir\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n        self.validator = ScientificValidator(config)\n        self.logger = configure_logging(\"autostudio.research\")\n\n    def _fallback(self, topic: str, sources: list[SourceRecord]) -> ResearchBundle:\n        facts = []\n        for index, source in enumerate(sources[:8], start=1):\n            claim = source.snippet.strip() or source.title\n            if claim:\n                facts.append(ResearchFact(\n                    fact_id=f\"F{index:02d}\", claim=claim[:500], source_ids=[source.source_id],\n                    confidence=0.55, visual_hint=source.title,\n                ))\n        return ResearchBundle(\n            topic=topic,\n            summary=\" \".join(fact.claim for fact in facts[:3])[:1200],\n            facts=facts,\n            references=sources,\n            visual_ideas=[source.title for source in sources[:5]],\n            limitations=[\"Fallback synthesis was used.\"],\n        )\n\n    def collect(self, topic: str, sources: list[SourceRecord], force_refresh: bool = False) -> tuple[ResearchBundle, ValidationReport]:\n        source_payload = [source.model_dump(mode=\"json\") for source in sources]\n        key = hash_value({\"topic\": topic, \"sources\": source_payload})\n        bundle_path = self.cache_dir / f\"{key}.json\"\n        validation_path = self.cache_dir / f\"{key}.validation.json\"\n        if bundle_path.exists() and validation_path.exists() and not force_refresh:\n            return ResearchBundle.model_validate(read_json(bundle_path)), ValidationReport.model_validate(read_json(validation_path))\n        prompt = f\"\"\"\nTopic: {topic}\n\nSource records:\n{json.dumps(source_payload, ensure_ascii=False, indent=2)}\n\nCreate a concise summary, 6-10 source-backed facts, important numbers, intuitive comparisons,\nflat-vector visual ideas, and limitations. Return:\n{{\"topic\":\"...\",\"summary\":\"...\",\"facts\":[{{\"fact_id\":\"F01\",\"claim\":\"...\",\"source_ids\":[\"source-id\"],\"confidence\":0.0,\"numeric_values\":[],\"visual_hint\":\"...\"}}],\"numbers\":[],\"comparisons\":[],\"visual_ideas\":[],\"limitations\":[]}}\nDo not return references; the application attaches them.\n\"\"\".strip()\n        try:\n            raw = self.llm.generate_json(self.SYSTEM_PROMPT, prompt, cache_namespace=\"research-synthesis\", max_new_tokens=2200, force_refresh=force_refresh)\n            raw.update({\"topic\": topic, \"references\": source_payload})\n            bundle = ResearchBundle.model_validate(raw)\n        except Exception as exc:\n            self.logger.warning(\"Research fallback: %s\", exc)\n            bundle = self._fallback(topic, sources)\n        bundle = bundle.model_copy(update={\"research_hash\": hash_value(bundle.model_dump(exclude={\"research_hash\"}))})\n        validation = self.validator.validate(bundle)\n        atomic_write_json(bundle_path, bundle)\n        atomic_write_json(validation_path, validation)\n        return bundle, validation\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 7. Script generator

Retention-first narration: hook → curiosity → explanation → escalation → ending, in timed beats. The LLM writes beats; Python enforces the word budget and computes speech timing.

In [ ]:
# Section 7 — retention-first script generator
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "script_generator.py": "from __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nfrom .config import StudioConfig\nfrom .hashing import atomic_write_json, hash_value, read_json\nfrom .llm import LLMClient\nfrom .logging_utils import configure_logging\nfrom .schemas import ResearchBundle, ScriptBeat, ScriptPackage, ValidationReport\n\n\nclass ScriptGenerator:\n    \"\"\"Retention-first narration. LLM writes beats; Python enforces timing and word budget.\"\"\"\n\n    SYSTEM_PROMPT = (\n        \"You write high-retention scientific animation scripts. Do not sound like Wikipedia. \"\n        \"Start inside a concrete event. Use short spoken sentences, contrast, consequences, \"\n        \"escalation, and payoff. Every factual beat must cite supplied fact IDs. Return JSON only.\"\n    )\n\n    def __init__(self, config: StudioConfig, llm: LLMClient, cache_dir: Path):\n        self.config = config\n        self.llm = llm\n        self.cache_dir = cache_dir\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n        self.logger = configure_logging(\"autostudio.script\")\n\n    def _timing(self, package: ScriptPackage) -> ScriptPackage:\n        beats, cursor, total = [], 0.0, 0\n        for beat in package.beats:\n            words = max(1, len(beat.narration.split()))\n            duration = words / self.config.script.words_per_second\n            beats.append(beat.model_copy(update={\n                \"words\": words, \"start_s\": round(cursor, 3),\n                \"end_s\": round(cursor + duration, 3), \"estimated_duration_s\": round(duration, 3),\n            }))\n            cursor += duration\n            total += words\n        return package.model_copy(update={\"beats\": beats, \"estimated_duration_s\": round(cursor, 3), \"total_words\": total})\n\n    def _repair(self, package: ScriptPackage, research: ResearchBundle) -> ScriptPackage:\n        package = self._timing(package)\n        minimum, maximum = self.config.script.target_words_min, self.config.script.target_words_max\n        beats = list(package.beats)\n        if package.total_words < minimum:\n            existing = {ref for beat in beats for ref in beat.evidence_refs}\n            for fact in [fact for fact in research.facts if fact.fact_id not in existing]:\n                if sum(len(beat.narration.split()) for beat in beats) >= minimum:\n                    break\n                beats.insert(max(1, len(beats) - 1), ScriptBeat(\n                    beat_id=f\"B{len(beats) + 1:02d}\", purpose=\"evidence-backed escalation\",\n                    narration=f\"Here is the strange part: {fact.claim}\", evidence_refs=[fact.fact_id],\n                ))\n        elif package.total_words > maximum:\n            repaired, running = [], 0\n            for beat in beats:\n                remaining = maximum - running\n                if remaining <= 0:\n                    break\n                narration = \" \".join(beat.narration.split()[:remaining])\n                repaired.append(beat.model_copy(update={\"narration\": narration}))\n                running += len(narration.split())\n            beats = repaired\n        return self._timing(package.model_copy(update={\"beats\": beats}))\n\n    def generate(self, research: ResearchBundle, validation: ValidationReport, force_refresh: bool = False) -> ScriptPackage:\n        key = hash_value({\n            \"research\": research.research_hash,\n            \"validation\": validation.model_dump(mode=\"json\"),\n            \"config\": self.config.script.model_dump(mode=\"json\"),\n        })\n        path = self.cache_dir / f\"{key}.json\"\n        if path.exists() and not force_refresh:\n            return ScriptPackage.model_validate(read_json(path))\n        evidence = {\n            \"summary\": research.summary,\n            \"facts\": [fact.model_dump(mode=\"json\") for fact in research.facts],\n            \"numbers\": [fact.model_dump(mode=\"json\") for fact in research.numbers],\n            \"comparisons\": research.comparisons,\n            \"limitations\": research.limitations,\n            \"validation\": validation.model_dump(mode=\"json\"),\n        }\n        prompt = f\"\"\"\nWrite an English scientific short about: {research.topic}\nTarget {self.config.script.target_words_min}-{self.config.script.target_words_max} spoken words,\n{self.config.script.scene_count_min}-{self.config.script.scene_count_max} beats, approximately {self.config.script.target_duration_seconds:.0f} seconds.\nUse cold open -> curiosity -> rule -> consequences -> escalation -> ending. No greeting or generic definition.\n\nEvidence:\n{json.dumps(evidence, ensure_ascii=False, indent=2)}\n\nReturn:\n{{\"topic\":\"...\",\"hook\":\"...\",\"curiosity\":\"...\",\"scientific_explanation\":\"...\",\"escalation\":\"...\",\"ending\":\"...\",\"tone\":\"curious, precise, escalating\",\"beats\":[{{\"beat_id\":\"B01\",\"purpose\":\"cold open\",\"narration\":\"...\",\"evidence_refs\":[\"F01\"]}}],\"estimated_duration_s\":0,\"total_words\":0}}\n\"\"\".strip()\n        raw = self.llm.generate_json(self.SYSTEM_PROMPT, prompt, cache_namespace=\"script\", max_new_tokens=2200, force_refresh=force_refresh)\n        raw[\"topic\"] = research.topic\n        package = self._repair(ScriptPackage.model_validate(raw), research)\n        if not (self.config.script.target_words_min <= package.total_words <= self.config.script.target_words_max):\n            repair_prompt = (\n                f\"Rewrite this ScriptPackage to {self.config.script.target_words_min}-{self.config.script.target_words_max} words, \"\n                f\"preserving evidence references and beat order. Return JSON only.\\n\\n{json.dumps(package.model_dump(mode='json'), ensure_ascii=False)}\"\n            )\n            try:\n                repaired = self.llm.generate_json(self.SYSTEM_PROMPT, repair_prompt, cache_namespace=\"script-repair\", max_new_tokens=1900, force_refresh=force_refresh)\n                repaired[\"topic\"] = research.topic\n                package = self._repair(ScriptPackage.model_validate(repaired), research)\n            except Exception as exc:\n                self.logger.warning(\"Script rewrite failed: %s\", exc)\n        package = package.model_copy(update={\"script_hash\": hash_value(package.model_dump(exclude={\"script_hash\"}))})\n        atomic_write_json(path, package)\n        return package\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 8. Storyboard generator

One scene per beat as structured JSON (objects, camera, animations, dashboard, motion, transition, asset requirements). Assets may only come from a fixed whitelist, which keeps the catalogue small and reusable. Keyword-mapped fallback if the LLM is unavailable.

In [ ]:
# Section 8 — storyboard generator
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "storyboard_generator.py": "from __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nfrom .config import StudioConfig\nfrom .hashing import atomic_write_json, hash_value, read_json\nfrom .llm import LLMClient\nfrom .logging_utils import configure_logging\nfrom .schemas import (\n    AnimationPlaceholder, AssetRequirement, CameraPlan, DashboardSpec,\n    ResearchBundle, Scene, SceneObject, ScriptPackage, Storyboard,\n)\n\n# The LLM may only request assets from this whitelist. This is what keeps the\n# asset catalogue small and reusable \u2014 no \"generate thousands of SVGs\".\nALLOWED_ASSET_TYPES = {\n    \"planet\", \"moon\", \"sun\", \"human\", \"cloud\", \"building\", \"arrow\", \"dashboard\",\n    \"wave\", \"tree\", \"animal\", \"rock\", \"star\", \"satellite\", \"black_hole\", \"atom\",\n    \"cell\", \"molecule\", \"volcano\", \"mountain\", \"tornado\", \"rocket\", \"battery\",\n    \"robot\", \"computer\", \"clock\", \"thermometer\", \"dna\", \"car\", \"airplane\",\n    \"shield\", \"generic_object\",\n}\n\n\nclass StoryboardGenerator:\n    \"\"\"Turns each narration beat into one visual scene. LLM picks semantic assets only.\"\"\"\n\n    SYSTEM_PROMPT = (\n        \"You direct a flat-vector scientific simulation channel. Translate every narration \"\n        \"beat into one clear visual consequence. Use only allowed asset types and a small \"\n        \"reusable set. Never request raster, textures, gradients, or arbitrary SVG code. \"\n        \"Return JSON only.\"\n    )\n\n    def __init__(self, config: StudioConfig, llm: LLMClient, cache_dir: Path):\n        self.config = config\n        self.llm = llm\n        self.cache_dir = cache_dir\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n        self.logger = configure_logging(\"autostudio.storyboard\")\n\n    def _keyword_assets(self, text: str) -> list[str]:\n        lowered = text.lower()\n        mapping = {\n            \"earth\": \"planet\", \"planet\": \"planet\", \"moon\": \"moon\", \"sun\": \"sun\",\n            \"human\": \"human\", \"people\": \"human\", \"person\": \"human\",\n            \"wind\": \"arrow\", \"atmosphere\": \"cloud\", \"cloud\": \"cloud\", \"ocean\": \"wave\",\n            \"water\": \"wave\", \"city\": \"building\", \"building\": \"building\", \"tree\": \"tree\",\n            \"animal\": \"animal\", \"rock\": \"rock\", \"asteroid\": \"rock\", \"star\": \"star\",\n            \"satellite\": \"satellite\", \"black hole\": \"black_hole\", \"atom\": \"atom\",\n            \"cell\": \"cell\", \"molecule\": \"molecule\", \"volcano\": \"volcano\",\n            \"mountain\": \"mountain\", \"tornado\": \"tornado\", \"rocket\": \"rocket\",\n            \"battery\": \"battery\", \"robot\": \"robot\", \"computer\": \"computer\",\n            \"clock\": \"clock\", \"temperature\": \"thermometer\", \"dna\": \"dna\",\n            \"car\": \"car\", \"airplane\": \"airplane\",\n        }\n        assets = []\n        for keyword, asset_type in mapping.items():\n            if keyword in lowered and asset_type not in assets:\n                assets.append(asset_type)\n        return assets[:4] or [\"generic_object\"]\n\n    def _fallback(self, script: ScriptPackage) -> Storyboard:\n        scenes = []\n        for index, beat in enumerate(script.beats, start=1):\n            requirements = [AssetRequirement(\n                asset_id=f\"{asset_type}-{index}-{asset_index}\", asset_type=asset_type,\n                label=asset_type.replace(\"_\", \" \").title(), source_prompt=beat.narration,\n                tags=[script.topic],\n            ) for asset_index, asset_type in enumerate(self._keyword_assets(beat.narration), start=1)]\n            objects = [SceneObject(\n                asset_id=req.asset_id, x=0.12 + 0.28 * (asset_index % 3),\n                y=0.30 + 0.18 * (asset_index // 3), width=0.28, height=0.28,\n                z_index=asset_index,\n            ) for asset_index, req in enumerate(requirements)]\n            scenes.append(Scene(\n                scene_id=f\"scene{index:02d}\", duration_s=max(2.5, beat.estimated_duration_s),\n                narration=beat.narration, title=beat.purpose.upper(), objects=objects,\n                camera=CameraPlan(shot=\"wide\", framing=\"center\"),\n                animations=[AnimationPlaceholder(target_asset_id=requirements[0].asset_id, kind=\"scale_in\", duration_s=0.8)],\n                dashboard=DashboardSpec(experiment_id=f\"EXPERIMENT #{index:03d}\", status_value=\"Running\", metric_label=\"PHASE\", metric_value=f\"{index}/{len(script.beats)}\"),\n                text=[beat.purpose.upper()], background=\"paper\", motion=\"hold\", transition=\"cut\",\n                asset_requirements=requirements,\n            ))\n        return Storyboard(\n            topic=script.topic, canvas_width=self.config.canvas.width, canvas_height=self.config.canvas.height,\n            style_name=self.config.style.name, scenes=scenes, estimated_duration_s=script.estimated_duration_s,\n        )\n\n    def generate(self, script: ScriptPackage, research: ResearchBundle, force_refresh: bool = False) -> Storyboard:\n        key = hash_value({\n            \"script\": script.script_hash, \"research\": research.research_hash,\n            \"canvas\": self.config.canvas.model_dump(mode=\"json\"), \"style\": self.config.style.model_dump(mode=\"json\"),\n        })\n        path = self.cache_dir / f\"{key}.json\"\n        if path.exists() and not force_refresh:\n            return Storyboard.model_validate(read_json(path))\n        prompt = f\"\"\"\nTopic: {script.topic}\nScript:\n{json.dumps(script.model_dump(mode='json'), ensure_ascii=False, indent=2)}\nVisual ideas:\n{json.dumps(research.visual_ideas, ensure_ascii=False)}\nAllowed asset types: {sorted(ALLOWED_ASSET_TYPES)}\n\nCreate one scene per beat. Coordinates/sizes are normalized 0-1. Use <=6 assets per scene and reuse IDs for recurring objects.\nVisual language: scientific experiment dashboard, flat geometry, strong headline, white/dark panels, blue accents, status cards, arrows, measurable consequences.\nReturn Storyboard JSON with scene_id, duration_s, narration, title, objects, camera, animations, dashboard, text, icons, background, motion, transition, and asset_requirements.\n\"\"\".strip()\n        try:\n            raw = self.llm.generate_json(self.SYSTEM_PROMPT, prompt, cache_namespace=\"storyboard\", max_new_tokens=3400, force_refresh=force_refresh)\n            raw.update({\n                \"topic\": script.topic, \"canvas_width\": self.config.canvas.width,\n                \"canvas_height\": self.config.canvas.height, \"style_name\": self.config.style.name,\n                \"estimated_duration_s\": script.estimated_duration_s,\n            })\n            storyboard = Storyboard.model_validate(raw)\n        except Exception as exc:\n            self.logger.warning(\"Storyboard fallback: %s\", exc)\n            storyboard = self._fallback(script)\n        storyboard = storyboard.model_copy(update={\"storyboard_hash\": hash_value(storyboard.model_dump(exclude={\"storyboard_hash\"}))})\n        atomic_write_json(path, storyboard)\n        return storyboard\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 9. Scene planner

Normalises the storyboard: clamps asset types to the whitelist, de-duplicates IDs, builds the global asset catalogue (exactly what to generate), and guarantees valid on-canvas placement inside the safe area.

In [ ]:
# Section 9 — scene planner + asset catalogue
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "scene_planner.py": "from __future__ import annotations\n\nfrom collections import OrderedDict\n\nfrom .config import StudioConfig\nfrom .hashing import hash_value\nfrom .logging_utils import configure_logging\nfrom .schemas import AssetRequirement, SceneObject, Storyboard\nfrom .storyboard_generator import ALLOWED_ASSET_TYPES\n\n\nclass ScenePlanner:\n    \"\"\"Normalises the storyboard: clamps asset types, dedupes IDs, builds the global\n    asset catalogue, and guarantees every requirement has a valid on-canvas placement.\"\"\"\n\n    def __init__(self, config: StudioConfig):\n        self.config = config\n        self.logger = configure_logging(\"autostudio.scene_planner\")\n\n    def _normalize_requirement(self, requirement: AssetRequirement) -> AssetRequirement:\n        asset_type = requirement.asset_type.lower().strip().replace(\" \", \"_\")\n        if asset_type not in ALLOWED_ASSET_TYPES:\n            asset_type = \"generic_object\"\n        asset_id = requirement.asset_id.strip() or f\"{asset_type}-{hash_value(requirement.source_prompt)[:8]}\"\n        return requirement.model_copy(update={\"asset_type\": asset_type, \"asset_id\": asset_id})\n\n    def _default_object(self, asset_id: str, index: int, count: int) -> SceneObject:\n        if count == 1:\n            return SceneObject(asset_id=asset_id, x=0.18, y=0.28, width=0.64, height=0.42)\n        columns = min(3, count)\n        row, col = divmod(index, columns)\n        return SceneObject(asset_id=asset_id, x=0.08 + col * (0.84 / columns), y=0.30 + row * 0.24, width=0.24, height=0.24, z_index=index)\n\n    def plan(self, storyboard: Storyboard) -> Storyboard:\n        catalog: OrderedDict[str, AssetRequirement] = OrderedDict()\n        scenes = []\n        for scene in storyboard.scenes:\n            requirements = [self._normalize_requirement(item) for item in scene.asset_requirements]\n            unique: OrderedDict[str, AssetRequirement] = OrderedDict()\n            for requirement in requirements:\n                unique.setdefault(requirement.asset_id, requirement)\n                catalog.setdefault(requirement.asset_id, requirement)\n            limited = list(unique.values())[:self.config.svg.max_scene_objects]\n            valid_ids = {item.asset_id for item in limited}\n            objects = [obj for obj in scene.objects if obj.asset_id in valid_ids]\n            existing = {obj.asset_id for obj in objects}\n            for index, requirement in enumerate(limited):\n                if requirement.asset_id not in existing:\n                    objects.append(self._default_object(requirement.asset_id, index, len(limited)))\n            clamped = []\n            for obj in objects[:self.config.svg.max_scene_objects]:\n                clamped.append(obj.model_copy(update={\n                    \"width\": max(0.04, min(obj.width, 1.0 - obj.x)),\n                    \"height\": max(0.04, min(obj.height, 1.0 - obj.y)),\n                }))\n            scenes.append(scene.model_copy(update={\"objects\": sorted(clamped, key=lambda item: item.z_index), \"asset_requirements\": limited}))\n        result = storyboard.model_copy(update={\n            \"scenes\": scenes, \"asset_catalog\": list(catalog.values()),\n            \"estimated_duration_s\": round(sum(scene.duration_s for scene in scenes), 3),\n        })\n        return result.model_copy(update={\"storyboard_hash\": hash_value(result.model_dump(exclude={\"storyboard_hash\"}))})\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 10. SVG generator

### Which SVG generation method — and why

| Approach | Deterministic | Editable vectors | Style-consistent | Cost / GPU | Verdict |
|---|---|---|---|---|---|
| **LLM writes raw SVG** | ✗ | ✗ | ✗ | tokens/GPU | rejected — not cacheable |
| **Raster model → vectorize** | ✗ | partial | ✗ | high GPU | rejected — raster origin, heavy |
| **HF vector models** | ✗ | partial | ✗ | GPU + immature | rejected — experimental |
| **Parametric Python templates** | ✓ | ✓ | ✓ | ~0 (CPU) | **chosen** |

The LLM decides *what* object a scene needs; a controlled Python template renders
*how* it looks. This yields byte-identical, reproducible, editable, flat-vector
output (no gradients/textures/raster) that is cacheable by content hash and is the
cleanest bridge to Motion Canvas / Remotion. Only assets the current storyboard
needs are generated; the rest are reused from cache.

In [ ]:
# Section 10 — deterministic SVG asset factory
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "svg_assets.py": "from __future__ import annotations\n\nimport math\nfrom pathlib import Path\nfrom typing import Callable\n\nimport svgwrite\n\nfrom .config import StyleConfig\nfrom .schemas import AssetRequirement\n\n\nclass AssetFactory:\n    \"\"\"Deterministic, flat-vector SVG primitives.\n\n    Each method emits a single 1000x1000 illustration built only from geometric\n    shapes, flat fills, and solid strokes (no gradients, textures, or raster).\n    The LLM never writes SVG; it only names an ``asset_type`` from a whitelist,\n    which makes every asset reproducible and cacheable by content hash.\n    \"\"\"\n\n    VERSION = \"svg-template-v1\"\n\n    def __init__(self, style: StyleConfig, viewbox: int = 1000):\n        self.style = style\n        self.palette = style.palette\n        self.viewbox = viewbox\n        self.stroke = style.stroke_width * 2.0\n\n    def _drawing(self) -> svgwrite.Drawing:\n        return svgwrite.Drawing(size=(\"100%\", \"100%\"), viewBox=f\"0 0 {self.viewbox} {self.viewbox}\")\n\n    def _planet(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing()\n        p = self.palette\n        dwg.add(dwg.circle(center=(500, 500), r=340, fill=p[\"white\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        for path in [\n            \"M280,350 C340,250 460,250 500,330 C450,380 390,410 330,430 Z\",\n            \"M520,270 C650,250 760,340 720,430 C650,390 610,350 520,360 Z\",\n            \"M440,520 C520,470 640,500 650,590 C590,650 520,700 450,650 Z\",\n            \"M240,520 C330,470 390,520 390,600 C330,640 280,610 240,570 Z\",\n        ]:\n            dwg.add(dwg.path(d=path, fill=p[\"secondary\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        return dwg\n\n    def _moon(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.circle(center=(500, 500), r=320, fill=p[\"white\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        for cx, cy, r in [(390, 380, 70), (610, 520, 95), (430, 650, 55), (640, 330, 40)]:\n            dwg.add(dwg.circle(center=(cx, cy), r=r, fill=p[\"paper\"], stroke=p[\"secondary\"], stroke_width=self.stroke * .65))\n        return dwg\n\n    def _sun(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        for angle in range(0, 360, 30):\n            rad = math.radians(angle)\n            dwg.add(dwg.line((500 + math.cos(rad) * 300, 500 + math.sin(rad) * 300), (500 + math.cos(rad) * 420, 500 + math.sin(rad) * 420), stroke=p[\"ink\"], stroke_width=self.stroke, stroke_linecap=\"round\"))\n        dwg.add(dwg.circle(center=(500, 500), r=220, fill=p[\"sun\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        return dwg\n\n    def _human(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.circle(center=(500, 260), r=105, fill=p[\"white\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        dwg.add(dwg.path(d=\"M380,420 Q500,350 620,420 L680,720 Q500,820 320,720 Z\", fill=p[\"secondary\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        for start, end in [((390, 470), (220, 650)), ((610, 470), (780, 650)), ((430, 720), (380, 930)), ((570, 720), (620, 930))]:\n            dwg.add(dwg.line(start, end, stroke=p[\"ink\"], stroke_width=self.stroke * 1.5, stroke_linecap=\"round\"))\n        for cx in (465, 535):\n            dwg.add(dwg.circle(center=(cx, 245), r=9, fill=p[\"ink\"]))\n        return dwg\n\n    def _cloud(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        group = dwg.g(fill=p[\"white\"], stroke=p[\"ink\"], stroke_width=self.stroke)\n        for cx, cy, r in [(320, 560, 150), (470, 450, 190), (650, 520, 165), (770, 600, 120)]:\n            group.add(dwg.circle(center=(cx, cy), r=r))\n        group.add(dwg.rect(insert=(250, 550), size=(560, 210), rx=100, ry=100)); dwg.add(group); return dwg\n\n    def _building(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.rect(insert=(260, 170), size=(480, 700), fill=p[\"secondary\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        for row in range(5):\n            for col in range(3):\n                dwg.add(dwg.rect(insert=(330 + col * 130, 250 + row * 110), size=(65, 65), fill=p[\"primary\"], stroke=p[\"ink\"], stroke_width=self.stroke * .55))\n        dwg.add(dwg.rect(insert=(445, 720), size=(110, 150), fill=p[\"paper\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    def _arrow(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.path(d=\"M120,430 H650 V250 L900,500 L650,750 V570 H120 Z\", fill=p[\"primary\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    def _wave(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.path(d=\"M80,620 C220,450 350,790 500,610 C650,430 790,740 930,560 L930,850 L80,850 Z\", fill=p[\"primary\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    def _tree(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.rect(insert=(450, 500), size=(100, 350), fill=p[\"secondary\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        for cx, cy, r in [(360, 430, 170), (520, 330, 200), (680, 450, 170)]:\n            dwg.add(dwg.circle(center=(cx, cy), r=r, fill=p[\"primary\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        return dwg\n\n    def _rock(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.polygon([(200, 650), (280, 350), (520, 180), (760, 330), (850, 650), (650, 840), (330, 820)], fill=p[\"secondary\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    def _black_hole(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.circle(center=(500, 500), r=190, fill=p[\"dark\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        for radius, width in [(280, 55), (350, 35)]:\n            dwg.add(dwg.ellipse(center=(500, 500), r=(radius, radius * .42), fill=\"none\", stroke=p[\"primary\"], stroke_width=width))\n        return dwg\n\n    def _atom(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        for rotation in (0, 60, 120):\n            dwg.add(dwg.ellipse(center=(500, 500), r=(360, 150), fill=\"none\", stroke=p[\"primary\"], stroke_width=self.stroke, transform=f\"rotate({rotation} 500 500)\"))\n        dwg.add(dwg.circle(center=(500, 500), r=85, fill=p[\"warning\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    def _rocket(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.path(d=\"M500,100 C690,260 680,650 500,800 C320,650 310,260 500,100 Z\", fill=p[\"white\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        dwg.add(dwg.circle(center=(500, 400), r=95, fill=p[\"primary\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        dwg.add(dwg.polygon([(430, 790), (500, 940), (570, 790)], fill=p[\"warning\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    def _battery(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.rect(insert=(210, 260), size=(580, 520), rx=45, ry=45, fill=p[\"white\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        dwg.add(dwg.rect(insert=(420, 180), size=(160, 90), fill=p[\"secondary\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        dwg.add(dwg.polygon([(520, 320), (390, 540), (500, 540), (450, 710), (620, 470), (510, 470)], fill=p[\"warning\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    def _computer(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.rect(insert=(140, 160), size=(720, 500), rx=35, ry=35, fill=p[\"white\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        dwg.add(dwg.rect(insert=(200, 220), size=(600, 370), fill=p[\"primary\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        dwg.add(dwg.rect(insert=(450, 660), size=(100, 160), fill=p[\"secondary\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    def _clock(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.circle(center=(500, 500), r=350, fill=p[\"white\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        dwg.add(dwg.line((500, 500), (500, 280), stroke=p[\"ink\"], stroke_width=self.stroke * 1.5))\n        dwg.add(dwg.line((500, 500), (690, 610), stroke=p[\"primary\"], stroke_width=self.stroke * 1.5)); return dwg\n\n    def _thermometer(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.rect(insert=(420, 160), size=(160, 560), rx=80, ry=80, fill=p[\"white\"], stroke=p[\"ink\"], stroke_width=self.stroke))\n        dwg.add(dwg.circle(center=(500, 760), r=170, fill=p[\"warning\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    def _star(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette; points = []\n        for index in range(10):\n            angle = math.radians(-90 + index * 36); radius = 330 if index % 2 == 0 else 145\n            points.append((500 + math.cos(angle) * radius, 500 + math.sin(angle) * radius))\n        dwg.add(dwg.polygon(points, fill=p[\"sun\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    def _generic_object(self, req: AssetRequirement) -> svgwrite.Drawing:\n        dwg = self._drawing(); p = self.palette\n        dwg.add(dwg.polygon([(250, 250), (700, 180), (840, 520), (650, 830), (250, 760), (130, 430)], fill=p[\"secondary\"], stroke=p[\"ink\"], stroke_width=self.stroke)); return dwg\n\n    # Types that map onto an existing primitive rather than owning a bespoke shape.\n    ALIASES = {\n        \"dashboard\": \"computer\", \"animal\": \"human\", \"satellite\": \"rocket\",\n        \"cell\": \"atom\", \"molecule\": \"atom\", \"volcano\": \"mountain\",\n        \"mountain\": \"rock\", \"tornado\": \"arrow\", \"robot\": \"computer\",\n        \"dna\": \"atom\", \"car\": \"generic_object\", \"airplane\": \"rocket\",\n        \"shield\": \"generic_object\",\n    }\n\n    def generate(self, requirement: AssetRequirement, output_path: Path) -> Path:\n        asset_type = self.ALIASES.get(requirement.asset_type, requirement.asset_type)\n        method: Callable[[AssetRequirement], svgwrite.Drawing] = getattr(self, f\"_{asset_type}\", self._generic_object)\n        drawing = method(requirement)\n        output_path.parent.mkdir(parents=True, exist_ok=True)\n        drawing.saveas(str(output_path), pretty=True)\n        return output_path\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 11. SVG cache

Content-addressed asset cache keyed by everything that changes the pixels (type, variant, palette, stroke, generator version) but not the scene — so the same asset is generated once and reused across scenes, runs, and future videos. Each asset stores full metadata (asset/prompt/SVG hashes, source prompt, topic, timestamps, embedding slot, reuse counter). Every SVG is validated (well-formed XML, no gradient/raster).

In [ ]:
# Section 11 — content-addressed asset cache
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "cache.py": "from __future__ import annotations\n\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom xml.etree import ElementTree as ET\n\nfrom .config import StudioConfig\nfrom .hashing import atomic_write_json, file_sha256, hash_value, read_json\nfrom .logging_utils import configure_logging\nfrom .schemas import AssetMetadata, AssetRequirement\nfrom .svg_assets import AssetFactory\n\n\nclass AssetCache:\n    \"\"\"Content-addressed SVG asset cache.\n\n    The asset hash is derived from everything that changes the pixels (type,\n    variant, palette, stroke, generator version) but NOT from the scene it\n    appears in \u2014 so the same \"planet\" is generated once and reused across\n    scenes and across future videos. Each hit bumps a reuse counter.\n    \"\"\"\n\n    def __init__(self, config: StudioConfig, cache_root: Path):\n        self.config = config\n        self.cache_root = cache_root / \"assets\"\n        self.cache_root.mkdir(parents=True, exist_ok=True)\n        self.index_path = self.cache_root / \"index.json\"\n        self.logger = configure_logging(\"autostudio.asset_cache\")\n        self.factory = AssetFactory(config.style, config.svg.asset_viewbox)\n\n    def _validate_svg(self, path: Path) -> None:\n        root = ET.parse(path).getroot()\n        if root.tag.split(\"}\")[-1] != \"svg\":\n            raise ValueError(f\"{path} is not an SVG document.\")\n        text = path.read_text(encoding=\"utf-8\").lower()\n        forbidden = [\"<lineargradient\", \"<radialgradient\", \"<image\", \"data:image/png\", \"data:image/jpeg\"]\n        violations = [token for token in forbidden if token in text]\n        if violations:\n            raise ValueError(f\"Forbidden SVG features: {violations}\")\n\n    def _update_index(self, asset_id: str, asset_hash: str) -> None:\n        index = read_json(self.index_path, {}) or {}\n        index[asset_id] = asset_hash\n        atomic_write_json(self.index_path, index)\n\n    def get_or_create(self, requirement: AssetRequirement, topic: str) -> tuple[Path, AssetMetadata]:\n        prompt_hash = hash_value(requirement.source_prompt)\n        asset_hash = hash_value({\n            \"asset_type\": requirement.asset_type, \"variant\": requirement.variant,\n            \"label\": requirement.label, \"palette\": self.config.style.palette,\n            \"stroke_width\": self.config.style.stroke_width,\n            \"generator_version\": self.config.svg.generator_version,\n        })\n        svg_path = self.cache_root / f\"{asset_hash}.svg\"\n        metadata_path = self.cache_root / f\"{asset_hash}.metadata.json\"\n        now = datetime.now(timezone.utc).isoformat()\n        if svg_path.exists() and metadata_path.exists() and self.config.cache.reuse_assets:\n            metadata = AssetMetadata.model_validate(read_json(metadata_path))\n            metadata = metadata.model_copy(update={\"reuse_counter\": metadata.reuse_counter + 1, \"updated_at\": now})\n            atomic_write_json(metadata_path, metadata)\n            self._update_index(requirement.asset_id, asset_hash)\n            self.logger.info(\"Cache hit %s (%s), reuse=%d\", requirement.asset_id, requirement.asset_type, metadata.reuse_counter)\n            return svg_path, metadata\n        self.factory.generate(requirement, svg_path)\n        if self.config.svg.validate_xml:\n            self._validate_svg(svg_path)\n        metadata = AssetMetadata(\n            asset_hash=asset_hash, prompt_hash=prompt_hash, svg_hash=file_sha256(svg_path),\n            asset_id=requirement.asset_id, asset_type=requirement.asset_type,\n            source_prompt=requirement.source_prompt, topic=topic, created_at=now,\n            updated_at=now, reuse_counter=0, embedding=None,\n            generator_version=self.config.svg.generator_version,\n        )\n        atomic_write_json(metadata_path, metadata)\n        self._update_index(requirement.asset_id, asset_hash)\n        self.logger.info(\"Generated %s (%s)\", requirement.asset_id, requirement.asset_type)\n        return svg_path, metadata\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 12. Scene composer

Composes cached SVG assets into a self-contained 1080×1920 scene: background, dashboard cards, headline, and every placed asset inlined. Also builds the storyboard preview (SVG contact sheet, HTML gallery, optional PNG).

In [ ]:
# Section 12 — scene composer + storyboard preview
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "composer.py": "from __future__ import annotations\n\nimport base64\nimport html\nimport re\nfrom pathlib import Path\n\nfrom .config import StudioConfig\nfrom .hashing import atomic_write_text\nfrom .logging_utils import configure_logging\nfrom .schemas import Scene\n\n\nclass SceneComposer:\n    \"\"\"Composes cached SVG assets into a full 1080x1920 static scene.\n\n    Output is one self-contained SVG per scene: background, dashboard cards,\n    headline, and every placed asset inlined (no external references). This is\n    the storyboard preview \u2014 deliberately static. Animation is Phase 2.\n    \"\"\"\n\n    def __init__(self, config: StudioConfig):\n        self.config = config\n        self.palette = config.style.palette\n        self.logger = configure_logging(\"autostudio.composer\")\n\n    def _inner_svg(self, path: Path) -> str:\n        text = path.read_text(encoding=\"utf-8\")\n        match = re.search(r\"<svg[^>]*>(.*)</svg>\", text, flags=re.DOTALL | re.IGNORECASE)\n        if not match:\n            raise ValueError(f\"Could not parse asset SVG: {path}\")\n        return re.sub(r'\\s+xmlns(:\\w+)?=\"[^\"]+\"', \"\", match.group(1))\n\n    def _wrap(self, text: str, max_chars: int = 21) -> list[str]:\n        lines, current = [], []\n        for word in text.split():\n            candidate = \" \".join(current + [word])\n            if len(candidate) > max_chars and current:\n                lines.append(\" \".join(current)); current = [word]\n            else:\n                current.append(word)\n        if current:\n            lines.append(\" \".join(current))\n        return lines[:4]\n\n    def _dashboard(self, scene: Scene, width: int, margin: int) -> str:\n        d = scene.dashboard; p = self.palette\n        font = self.config.style.font_family\n        metric_fill = p[\"dark\"] if scene.background == \"dark\" else p[\"paper\"]\n        metric_text = p[\"white\"] if scene.background == \"dark\" else p[\"ink\"]\n        marker = p[\"warning\"] if d.severity in {\"critical\", \"warning\"} else p[\"primary\"]\n        return (\n            f'<g id=\"dashboard\">'\n            f'<rect x=\"{margin}\" y=\"{margin}\" width=\"300\" height=\"120\" fill=\"{p[\"paper\"]}\" stroke=\"{p[\"ink\"]}\" stroke-width=\"4\"/>'\n            f'<text x=\"{margin + 18}\" y=\"{margin + 30}\" font-family=\"{font}\" font-size=\"20\" font-weight=\"700\" fill=\"{p[\"ink\"]}\">{html.escape(d.experiment_id)}</text>'\n            f'<text x=\"{margin + 18}\" y=\"{margin + 60}\" font-family=\"{font}\" font-size=\"17\" fill=\"{p[\"ink\"]}\">{html.escape(d.status_label)}:</text>'\n            f'<text x=\"{margin + 18}\" y=\"{margin + 90}\" font-family=\"{font}\" font-size=\"22\" font-weight=\"700\" fill=\"{p[\"ink\"]}\">{html.escape(d.status_value)}</text>'\n            f'<rect x=\"{width - margin - 300}\" y=\"{margin}\" width=\"300\" height=\"100\" fill=\"{metric_fill}\" stroke=\"{p[\"ink\"]}\" stroke-width=\"4\"/>'\n            f'<polygon points=\"{width - margin - 285},{margin + 78} {width - margin - 270},{margin + 42} {width - margin - 255},{margin + 78}\" fill=\"{marker}\"/>'\n            f'<text x=\"{width - margin - 235}\" y=\"{margin + 35}\" font-family=\"{font}\" font-size=\"18\" font-weight=\"700\" fill=\"{metric_text}\">{html.escape(d.metric_label)}</text>'\n            f'<text x=\"{width - margin - 235}\" y=\"{margin + 74}\" font-family=\"{font}\" font-size=\"30\" font-weight=\"800\" fill=\"{metric_text}\">{html.escape(d.metric_value)}</text>'\n            f'</g>'\n        )\n\n    def compose_scene(self, scene: Scene, asset_paths: dict[str, Path], output_path: Path, canvas_width: int, canvas_height: int) -> Path:\n        p = self.palette; margin = self.config.canvas.safe_margin\n        font = self.config.style.font_family\n        background = p[\"dark\"] if scene.background == \"dark\" else p[\"paper\"]\n        foreground = p[\"white\"] if scene.background == \"dark\" else p[\"ink\"]\n        groups = []\n        for obj in sorted(scene.objects, key=lambda item: item.z_index):\n            path = asset_paths.get(obj.asset_id)\n            if not path:\n                continue\n            x, y, w, h = obj.x * canvas_width, obj.y * canvas_height, obj.width * canvas_width, obj.height * canvas_height\n            scale = self.config.svg.asset_viewbox\n            groups.append(\n                f'<g id=\"{html.escape(obj.asset_id)}\" opacity=\"{obj.opacity}\" '\n                f'transform=\"translate({x + w / 2:.2f} {y + h / 2:.2f}) rotate({obj.rotation:.2f}) '\n                f'translate({-w / 2:.2f} {-h / 2:.2f}) scale({w / scale:.6f} {h / scale:.6f})\">'\n                f'{self._inner_svg(path)}</g>'\n            )\n        lines = self._wrap(scene.title or (scene.text[0] if scene.text else \"\"))\n        tspans = \"\".join(\n            f'<tspan x=\"{canvas_width / 2:.1f}\" dy=\"{0 if i == 0 else 72}\">{html.escape(line.upper())}</tspan>'\n            for i, line in enumerate(lines)\n        )\n        svg = (\n            f'<?xml version=\"1.0\" encoding=\"UTF-8\"?>\\n'\n            f'<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"{canvas_width}\" height=\"{canvas_height}\" viewBox=\"0 0 {canvas_width} {canvas_height}\">\\n'\n            f'<rect width=\"{canvas_width}\" height=\"{canvas_height}\" fill=\"{background}\"/>{self._dashboard(scene, canvas_width, margin)}\\n'\n            f'<text x=\"{canvas_width / 2}\" y=\"220\" text-anchor=\"middle\" font-family=\"{font}\" font-size=\"64\" font-weight=\"800\" fill=\"{foreground}\" letter-spacing=\"-1.5\">{tspans}</text>\\n'\n            f'{\"\".join(groups)}\\n'\n            f'<text x=\"{margin}\" y=\"{canvas_height - margin}\" font-family=\"{font}\" font-size=\"20\" fill=\"{foreground}\" opacity=\"0.65\">{html.escape(scene.scene_id.upper())}</text>\\n'\n            f'</svg>'\n        )\n        output_path.parent.mkdir(parents=True, exist_ok=True)\n        atomic_write_text(output_path, svg)\n        return output_path\n\n    def compose_contact_sheet(self, scene_paths: list[Path], output_path: Path, columns: int = 3, thumb_width: int = 360, thumb_height: int = 640, gap: int = 16) -> Path:\n        \"\"\"A single SVG grid embedding each scene as a data-URI thumbnail.\"\"\"\n        rows = (len(scene_paths) + columns - 1) // columns\n        width = columns * thumb_width + (columns + 1) * gap\n        height = rows * thumb_height + (rows + 1) * gap\n        images = []\n        for index, path in enumerate(scene_paths):\n            row, col = divmod(index, columns)\n            x = gap + col * (thumb_width + gap)\n            y = gap + row * (thumb_height + gap)\n            encoded = base64.b64encode(path.read_bytes()).decode(\"ascii\")\n            images.append(f'<image x=\"{x}\" y=\"{y}\" width=\"{thumb_width}\" height=\"{thumb_height}\" href=\"data:image/svg+xml;base64,{encoded}\"/>')\n        atomic_write_text(output_path, f'<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"{width}\" height=\"{height}\" viewBox=\"0 0 {width} {height}\"><rect width=\"{width}\" height=\"{height}\" fill=\"#D7DBDD\"/>{\"\".join(images)}</svg>')\n        return output_path\n\n    def compose_preview_html(self, scene_paths: list[Path], output_path: Path) -> Path:\n        \"\"\"Self-contained HTML gallery of the storyboard (openable in any browser).\"\"\"\n        cards = []\n        for path in scene_paths:\n            encoded = base64.b64encode(path.read_bytes()).decode(\"ascii\")\n            cards.append(f'<article><img src=\"data:image/svg+xml;base64,{encoded}\"><p>{html.escape(path.name)}</p></article>')\n        document = (\n            '<!doctype html><html><head><meta charset=\"utf-8\"><title>Storyboard Preview</title>'\n            '<style>body{font-family:Arial;background:#eceff1;padding:24px}'\n            'main{display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:18px}'\n            'article{background:white;border:1px solid #b0bec5;padding:10px}'\n            'img{width:100%}p{font-weight:700}</style></head><body><main>'\n            f'{\"\".join(cards)}</main></body></html>'\n        )\n        atomic_write_text(output_path, document)\n        return output_path\n\n    def rasterize(self, svg_path: Path, output_path: Path, output_width: int | None = None) -> Path | None:\n        \"\"\"Optional CairoSVG rasterization for reliable inline previews.\n\n        Returns None (and logs) if CairoSVG/Cairo is unavailable, so the pipeline\n        never hard-fails just because a preview couldn't be rasterized.\n        \"\"\"\n        try:\n            import cairosvg\n        except Exception as exc:\n            self.logger.warning(\"CairoSVG unavailable, skipping raster preview: %s\", exc)\n            return None\n        try:\n            cairosvg.svg2png(url=str(svg_path), write_to=str(output_path), output_width=output_width)\n            return output_path\n        except Exception as exc:\n            self.logger.warning(\"Rasterization failed for %s: %s\", svg_path, exc)\n            return None\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 13. Voice-over and audio timeline

`VoiceoverEngine` synthesises per-scene narration with **Edge TTS** (free,
online) and falls back to **espeak-ng** offline; clips are content-addressed and
never re-synthesised. The audio duration is authoritative — each scene stretches
to match its clip. `AudioMixer` builds a procedural music bed and per-beat sound
cues with FFmpeg `lavfi`, side-chain ducks the music under the voice, and
loudness-normalises the mix. No external audio assets, no licensing.

In [ ]:
# Section 13 — voice-over, procedural music, mixing
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "audio.py": "from __future__ import annotations\n\nimport asyncio\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\nfrom .config import StudioConfig\nfrom .hashing import atomic_write_json, file_sha256, hash_value, read_json\nfrom .logging_utils import configure_logging\nfrom .schemas import AudioClip, AudioTimeline, Storyboard\n\ntry:  # Allows re-entrant event loops inside notebooks; optional at import time.\n    import nest_asyncio\n\n    nest_asyncio.apply()\nexcept Exception:  # pragma: no cover - nest_asyncio missing outside notebooks\n    pass\n\n\ndef run_command(command: list[str], *, check: bool = True) -> subprocess.CompletedProcess:\n    \"\"\"Thin, loud subprocess wrapper. Surfaces the tail of stderr on failure.\"\"\"\n    result = subprocess.run(command, capture_output=True, text=True)\n    if check and result.returncode != 0:\n        raise RuntimeError(f\"Command failed: {' '.join(command)}\\n{result.stderr[-4000:]}\")\n    return result\n\n\ndef media_duration(path: Path) -> float:\n    result = run_command([\n        \"ffprobe\", \"-v\", \"error\", \"-show_entries\", \"format=duration\",\n        \"-of\", \"default=noprint_wrappers=1:nokey=1\", str(path),\n    ])\n    return float(result.stdout.strip())\n\n\nclass VoiceoverEngine:\n    \"\"\"Edge TTS (free, online) with an offline espeak-ng fallback. Clips are\n    content-addressed, so identical narration is never re-synthesised.\"\"\"\n\n    def __init__(self, config: StudioConfig, cache_root: Path):\n        self.config = config\n        self.cache_dir = cache_root / \"audio\"\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n        self.logger = configure_logging(\"autostudio.voice\")\n\n    async def _edge_tts(self, text: str, output_path: Path) -> None:\n        import edge_tts\n\n        communicate = edge_tts.Communicate(\n            text=text, voice=self.config.voice.voice,\n            rate=self.config.voice.rate, pitch=self.config.voice.pitch,\n        )\n        await communicate.save(str(output_path))\n\n    def _run_async(self, coroutine) -> None:\n        try:\n            loop = asyncio.get_event_loop()\n            loop.run_until_complete(coroutine)\n        except RuntimeError:\n            asyncio.run(coroutine)\n\n    def _synthesize_edge(self, text: str, output_path: Path) -> None:\n        last_error: Exception | None = None\n        for _ in range(2):\n            try:\n                self._run_async(self._edge_tts(text, output_path))\n                if output_path.exists() and output_path.stat().st_size >= 1000:\n                    return\n                raise RuntimeError(\"Edge TTS produced no valid audio.\")\n            except Exception as exc:\n                last_error = exc\n        raise RuntimeError(f\"Edge TTS failed after two attempts: {last_error}\")\n\n    def _synthesize_local(self, text: str, output_path: Path) -> None:\n        if not shutil.which(\"espeak-ng\"):\n            raise RuntimeError(\"espeak-ng is not installed.\")\n        wav_path = output_path.with_suffix(\".wav\")\n        run_command([\n            \"espeak-ng\", \"-v\", self.config.voice.local_voice,\n            \"-s\", str(self.config.voice.local_words_per_minute),\n            \"-w\", str(wav_path), text,\n        ])\n        run_command([\n            \"ffmpeg\", \"-y\", \"-i\", str(wav_path), \"-ar\", str(self.config.voice.sample_rate),\n            \"-c:a\", \"aac\", \"-b:a\", \"160k\", str(output_path),\n        ])\n        wav_path.unlink(missing_ok=True)\n\n    def synthesize_clip(self, scene_id: str, text: str) -> tuple[Path, str, str]:\n        key = hash_value({\n            \"text\": text, \"voice\": self.config.voice.voice,\n            \"rate\": self.config.voice.rate, \"pitch\": self.config.voice.pitch,\n            \"tail\": self.config.voice.scene_tail_silence,\n        })\n        final_path = self.cache_dir / f\"{key}.m4a\"\n        metadata_path = self.cache_dir / f\"{key}.json\"\n        if final_path.exists() and metadata_path.exists():\n            metadata = read_json(metadata_path, {})\n            return final_path, str(metadata.get(\"provider\", \"cache\")), key\n\n        raw_path = self.cache_dir / f\"{key}.raw.mp3\"\n        provider = \"edge-tts\"\n        try:\n            self._synthesize_edge(text, raw_path)\n        except Exception as exc:\n            self.logger.warning(\"Edge TTS failed for %s: %s. Using espeak-ng.\", scene_id, exc)\n            provider = \"espeak-ng\"\n            raw_path = self.cache_dir / f\"{key}.raw.m4a\"\n            self._synthesize_local(text, raw_path)\n\n        run_command([\n            \"ffmpeg\", \"-y\", \"-i\", str(raw_path),\n            \"-af\", f\"apad=pad_dur={self.config.voice.scene_tail_silence},aresample={self.config.voice.sample_rate}\",\n            \"-c:a\", \"aac\", \"-b:a\", \"160k\", str(final_path),\n        ])\n        raw_path.unlink(missing_ok=True)\n        atomic_write_json(metadata_path, {\n            \"provider\": provider, \"text\": text, \"audio_hash\": key,\n            \"duration_s\": media_duration(final_path), \"file_sha256\": file_sha256(final_path),\n        })\n        return final_path, provider, key\n\n    def build_timeline(self, storyboard: Storyboard, run_dir: Path) -> tuple[AudioTimeline, Storyboard]:\n        audio_dir = run_dir / \"audio\"\n        audio_dir.mkdir(parents=True, exist_ok=True)\n        clips: list[AudioClip] = []\n        updated_scenes = []\n        cursor = 0.0\n        clip_paths: list[Path] = []\n\n        for scene in storyboard.scenes:\n            cached_path, provider, audio_hash = self.synthesize_clip(scene.scene_id, scene.narration)\n            destination = audio_dir / f\"{scene.scene_id}.m4a\"\n            if not destination.exists() or file_sha256(destination) != file_sha256(cached_path):\n                shutil.copy2(cached_path, destination)\n            duration = media_duration(destination)\n            clips.append(AudioClip(\n                scene_id=scene.scene_id, text=scene.narration, path=str(destination),\n                start_s=round(cursor, 3), end_s=round(cursor + duration, 3),\n                duration_s=round(duration, 3), provider=provider, audio_hash=audio_hash,\n            ))\n            # The audio duration is authoritative: the scene stretches to match it.\n            updated_scenes.append(scene.model_copy(update={\"duration_s\": round(duration, 3)}))\n            cursor += duration\n            clip_paths.append(destination)\n\n        concat_path = audio_dir / \"voice_concat.txt\"\n        concat_path.write_text(\"\\n\".join(f\"file '{path.as_posix()}'\" for path in clip_paths), encoding=\"utf-8\")\n        voice_track = audio_dir / \"voice_track.m4a\"\n        run_command([\n            \"ffmpeg\", \"-y\", \"-f\", \"concat\", \"-safe\", \"0\", \"-i\", str(concat_path),\n            \"-c:a\", \"aac\", \"-b:a\", \"192k\", \"-ar\", str(self.config.voice.sample_rate), str(voice_track),\n        ])\n        actual_duration = media_duration(voice_track)\n        timeline = AudioTimeline(\n            clips=clips, voice_track=str(voice_track), duration_s=round(actual_duration, 3),\n            timeline_hash=hash_value([clip.model_dump(mode=\"json\") for clip in clips]),\n        )\n        updated_storyboard = storyboard.model_copy(update={\"scenes\": updated_scenes, \"estimated_duration_s\": round(actual_duration, 3)})\n        atomic_write_json(audio_dir / \"timeline.json\", timeline)\n        return timeline, updated_storyboard\n\n\nclass AudioMixer:\n    \"\"\"Procedural music bed + per-beat sound cues, side-chain ducked under the\n    voice and loudness-normalised. Everything is synthesised with FFmpeg lavfi \u2014\n    no external audio assets, no licensing.\"\"\"\n\n    def __init__(self, config: StudioConfig):\n        self.config = config\n        self.logger = configure_logging(\"autostudio.audio_mixer\")\n\n    def build_music_bed(self, duration: float, output_path: Path) -> Path:\n        fade_out = max(0.0, duration - 2.0)\n        run_command([\n            \"ffmpeg\", \"-y\",\n            \"-f\", \"lavfi\", \"-i\", f\"sine=frequency=110:sample_rate=48000:duration={duration}\",\n            \"-f\", \"lavfi\", \"-i\", f\"sine=frequency=220:sample_rate=48000:duration={duration}\",\n            \"-f\", \"lavfi\", \"-i\", f\"anoisesrc=color=pink:sample_rate=48000:duration={duration}\",\n            \"-filter_complex\",\n            f\"[0:a]volume=0.035[a0];[1:a]volume=0.018[a1];\"\n            f\"[2:a]volume=0.006,highpass=f=100,lowpass=f=1200[a2];\"\n            f\"[a0][a1][a2]amix=inputs=3:normalize=0,\"\n            f\"afade=t=in:st=0:d=1.2,afade=t=out:st={fade_out}:d=2,alimiter=limit=0.85[a]\",\n            \"-map\", \"[a]\", \"-c:a\", \"aac\", \"-b:a\", \"128k\", str(output_path),\n        ])\n        return output_path\n\n    def build_sfx_track(self, timeline: AudioTimeline, output_path: Path) -> Path:\n        duration = timeline.duration_s\n        inputs: list[str] = []\n        filter_parts: list[str] = []\n        labels: list[str] = []\n        for index, clip in enumerate(timeline.clips):\n            delay = max(0, int(clip.start_s * 1000))\n            frequency = 520 if index % 3 == 0 else 360\n            inputs.extend([\"-f\", \"lavfi\", \"-i\", f\"sine=frequency={frequency}:sample_rate=48000:duration=0.10\"])\n            label = f\"s{index}\"\n            filter_parts.append(f\"[{index}:a]volume=0.10,afade=t=out:st=0.03:d=0.07,adelay={delay}|{delay}[{label}]\")\n            labels.append(f\"[{label}]\")\n        if not labels:\n            run_command([\"ffmpeg\", \"-y\", \"-f\", \"lavfi\", \"-i\", f\"anullsrc=r=48000:cl=stereo:d={duration}\", str(output_path)])\n            return output_path\n        filter_parts.append(\"\".join(labels) + f\"amix=inputs={len(labels)}:normalize=0,apad=pad_dur={duration}[mix]\")\n        run_command([\n            \"ffmpeg\", \"-y\", *inputs, \"-filter_complex\", \";\".join(filter_parts),\n            \"-map\", \"[mix]\", \"-t\", str(duration), \"-c:a\", \"aac\", \"-b:a\", \"128k\", str(output_path),\n        ])\n        return output_path\n\n    def mix(self, timeline: AudioTimeline, run_dir: Path) -> Path:\n        audio_dir = run_dir / \"audio\"\n        audio_dir.mkdir(parents=True, exist_ok=True)\n        voice = Path(timeline.voice_track)\n        duration = timeline.duration_s\n        music = audio_dir / \"background_music.m4a\"\n        sfx = audio_dir / \"sound_cues.m4a\"\n        final_audio = audio_dir / \"final_audio.m4a\"\n\n        if self.config.audio.background_enabled:\n            self.build_music_bed(duration, music)\n        else:\n            run_command([\"ffmpeg\", \"-y\", \"-f\", \"lavfi\", \"-i\", f\"anullsrc=r=48000:cl=stereo:d={duration}\", str(music)])\n        if self.config.audio.sfx_enabled:\n            self.build_sfx_track(timeline, sfx)\n        else:\n            run_command([\"ffmpeg\", \"-y\", \"-f\", \"lavfi\", \"-i\", f\"anullsrc=r=48000:cl=stereo:d={duration}\", str(sfx)])\n\n        run_command([\n            \"ffmpeg\", \"-y\", \"-i\", str(voice), \"-i\", str(music), \"-i\", str(sfx),\n            \"-filter_complex\",\n            f\"[1:a]volume={self.config.audio.music_volume}[music];\"\n            f\"[2:a]volume={self.config.audio.sfx_volume}[sfx];\"\n            f\"[music][0:a]sidechaincompress=threshold=0.025:ratio=8:attack=20:release=250[ducked];\"\n            f\"[0:a][ducked][sfx]amix=inputs=3:weights=1 1 1:normalize=0,\"\n            f\"loudnorm=I={self.config.audio.voice_loudness_lufs}:TP=-1.5:LRA=9[a]\",\n            \"-map\", \"[a]\", \"-t\", str(duration), \"-c:a\", \"aac\", \"-b:a\", \"192k\", str(final_audio),\n        ])\n        return final_audio\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 14. Safe-zone subtitles

Word-timed captions in both **SRT** and safe-zone **ASS** (styled, positioned above the mobile UI). Narration is chunked into short karaoke-style lines and each clip's duration is distributed by word count.

In [ ]:
# Section 14 — safe-zone subtitles
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "captions.py": "from __future__ import annotations\n\nimport re\nfrom pathlib import Path\n\nfrom .config import StudioConfig\nfrom .hashing import atomic_write_json, atomic_write_text\nfrom .schemas import AudioTimeline, CaptionSegment\n\n\ndef srt_time(seconds: float) -> str:\n    total_ms = int(round(seconds * 1000))\n    hours, remainder = divmod(total_ms, 3_600_000)\n    minutes, remainder = divmod(remainder, 60_000)\n    secs, milliseconds = divmod(remainder, 1000)\n    return f\"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}\"\n\n\ndef ass_time(seconds: float) -> str:\n    centiseconds = int(round(seconds * 100))\n    hours, remainder = divmod(centiseconds, 360_000)\n    minutes, remainder = divmod(remainder, 6_000)\n    secs, cs = divmod(remainder, 100)\n    return f\"{hours}:{minutes:02d}:{secs:02d}.{cs:02d}\"\n\n\nclass CaptionGenerator:\n    \"\"\"Word-timed captions in both SRT and safe-zone ASS. Chunks narration into\n    short karaoke-style lines and distributes each clip's duration by word count.\"\"\"\n\n    def __init__(self, config: StudioConfig):\n        self.config = config\n\n    def _chunks(self, text: str) -> list[str]:\n        words = text.split()\n        chunks: list[str] = []\n        current: list[str] = []\n        for word in words:\n            current.append(word)\n            joined = \" \".join(current)\n            if (\n                len(current) >= self.config.captions.max_words\n                or len(joined) >= self.config.captions.max_chars\n                or re.search(r\"[.!?]$\", word)\n            ):\n                chunks.append(joined)\n                current = []\n        if current:\n            chunks.append(\" \".join(current))\n        return chunks\n\n    def segments(self, timeline: AudioTimeline) -> list[CaptionSegment]:\n        output: list[CaptionSegment] = []\n        index = 1\n        for clip in timeline.clips:\n            chunks = self._chunks(clip.text)\n            weights = [max(1, len(chunk.split())) for chunk in chunks]\n            total = max(1, sum(weights))\n            cursor = clip.start_s\n            for chunk, weight in zip(chunks, weights):\n                chunk_duration = clip.duration_s * weight / total\n                output.append(CaptionSegment(\n                    index=index, text=chunk, start_s=round(cursor, 3),\n                    end_s=round(min(clip.end_s, cursor + chunk_duration), 3), scene_id=clip.scene_id,\n                ))\n                cursor += chunk_duration\n                index += 1\n        return output\n\n    def write(self, timeline: AudioTimeline, run_dir: Path) -> tuple[Path, Path, list[CaptionSegment]]:\n        caption_dir = run_dir / \"captions\"\n        caption_dir.mkdir(parents=True, exist_ok=True)\n        segments = self.segments(timeline)\n\n        srt_lines: list[str] = []\n        for segment in segments:\n            srt_lines.extend([\n                str(segment.index),\n                f\"{srt_time(segment.start_s)} --> {srt_time(segment.end_s)}\",\n                segment.text, \"\",\n            ])\n        srt_path = caption_dir / \"captions.srt\"\n        atomic_write_text(srt_path, \"\\n\".join(srt_lines))\n\n        cfg = self.config.captions\n        ass_header = f\"\"\"[Script Info]\nScriptType: v4.00+\nPlayResX: {self.config.render.width}\nPlayResY: {self.config.render.height}\nWrapStyle: 2\nScaledBorderAndShadow: yes\n\n[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\nStyle: Default,{cfg.font_name},{cfg.font_size},{cfg.primary_color},&H0000D7FF,{cfg.outline_color},{cfg.back_color},-1,0,0,0,100,100,0,0,3,2,0,2,{cfg.margin_left},{cfg.margin_right},{cfg.margin_vertical},1\n\n[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n\"\"\"\n        events = []\n        for segment in segments:\n            safe = segment.text.replace(\"{\", \"(\").replace(\"}\", \")\")\n            events.append(\n                f\"Dialogue: 0,{ass_time(segment.start_s)},{ass_time(segment.end_s)},Default,,0,0,0,,\"\n                f\"{{\\\\fad(60,60)}}{safe}\"\n            )\n        ass_path = caption_dir / \"captions.ass\"\n        atomic_write_text(ass_path, ass_header + \"\\n\".join(events))\n        atomic_write_json(caption_dir / \"segments.json\", [segment.model_dump(mode=\"json\") for segment in segments])\n        return srt_path, ass_path, segments\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 15. Animated video rendering

`VideoRenderer`: static scene SVG → PNG (CairoSVG) → per-scene motion clip
(FFmpeg `zoompan`: zoom/pan/pulse per `scene.motion`) → concatenated visuals →
muxed with the final audio and **burned-in ASS captions** → `short.mp4`. Rasters
and scene clips are cached by content hash; the output is validated with
`ffprobe` (resolution, fps, duration, audio stream) and the run fails loudly if
any check fails.

In [ ]:
# Section 15 — CairoSVG + FFmpeg renderer with ffprobe validation
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "renderer.py": "from __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport cairosvg\n\nfrom .audio import run_command\nfrom .config import StudioConfig\nfrom .hashing import atomic_write_json, file_sha256, hash_value\nfrom .logging_utils import configure_logging\nfrom .schemas import RenderReport, Storyboard\n\n\nclass VideoRenderer:\n    \"\"\"Static scene SVG -> PNG (CairoSVG) -> per-scene motion clip (FFmpeg\n    zoompan) -> concatenated visuals -> muxed with audio + burned-in captions.\n    Rasters and scene clips are cached by content hash; the final output is\n    validated with ffprobe (resolution, fps, duration, audio stream).\"\"\"\n\n    def __init__(self, config: StudioConfig, cache_root: Path):\n        self.config = config\n        self.raster_cache = cache_root / \"raster\"\n        self.clip_cache = cache_root / \"scene_clips\"\n        self.raster_cache.mkdir(parents=True, exist_ok=True)\n        self.clip_cache.mkdir(parents=True, exist_ok=True)\n        self.logger = configure_logging(\"autostudio.renderer\")\n\n    def rasterize(self, svg_path: Path) -> Path:\n        key = hash_value({\n            \"svg_hash\": file_sha256(svg_path),\n            \"width\": self.config.render.width, \"height\": self.config.render.height,\n        })\n        png_path = self.raster_cache / f\"{key}.png\"\n        if not png_path.exists():\n            cairosvg.svg2png(\n                url=str(svg_path), write_to=str(png_path),\n                output_width=self.config.render.width, output_height=self.config.render.height,\n            )\n        return png_path\n\n    def _motion_filter(self, motion: str, frames: int) -> str:\n        width = self.config.render.width\n        height = self.config.render.height\n        frames = max(1, frames)\n        motion = (motion or \"hold\").lower()\n        if motion == \"zoom_in\":\n            z = \"min(zoom+0.0009,1.12)\"; x = \"(iw-iw/zoom)/2\"; y = \"(ih-ih/zoom)/2\"\n        elif motion == \"zoom_out\":\n            z = \"if(eq(on,1),1.12,max(zoom-0.0009,1.0))\"; x = \"(iw-iw/zoom)/2\"; y = \"(ih-ih/zoom)/2\"\n        elif motion == \"pan_left\":\n            z = \"1.08\"; x = f\"(iw-iw/zoom)*(1-on/{frames})\"; y = \"(ih-ih/zoom)/2\"\n        elif motion == \"pan_right\":\n            z = \"1.08\"; x = f\"(iw-iw/zoom)*(on/{frames})\"; y = \"(ih-ih/zoom)/2\"\n        elif motion == \"pulse\":\n            z = \"1.035+0.018*sin(on/7)\"; x = \"(iw-iw/zoom)/2\"; y = \"(ih-ih/zoom)/2\"\n        else:\n            z = \"min(zoom+0.00018,1.025)\"; x = \"(iw-iw/zoom)/2\"; y = \"(ih-ih/zoom)/2\"\n        fade_out_start = max(0.0, frames / self.config.render.fps - self.config.render.scene_fade_seconds)\n        return (\n            f\"scale={width}:{height}:force_original_aspect_ratio=increase,\"\n            f\"crop={width}:{height},\"\n            f\"zoompan=z='{z}':x='{x}':y='{y}':d={frames}:s={width}x{height}:fps={self.config.render.fps},\"\n            f\"fade=t=in:st=0:d={self.config.render.scene_fade_seconds},\"\n            f\"fade=t=out:st={fade_out_start}:d={self.config.render.scene_fade_seconds},\"\n            \"format=yuv420p\"\n        )\n\n    def make_scene_clip(self, scene, png_path: Path) -> Path:\n        frames = max(1, int(round(scene.duration_s * self.config.render.fps)))\n        key = hash_value({\n            \"png_hash\": file_sha256(png_path), \"duration\": round(scene.duration_s, 3),\n            \"motion\": scene.motion, \"fps\": self.config.render.fps, \"crf\": self.config.render.crf,\n        })\n        clip_path = self.clip_cache / f\"{key}.mp4\"\n        if clip_path.exists():\n            return clip_path\n        run_command([\n            \"ffmpeg\", \"-y\", \"-loop\", \"1\", \"-i\", str(png_path),\n            \"-vf\", self._motion_filter(scene.motion, frames),\n            \"-frames:v\", str(frames), \"-an\", \"-c:v\", \"libx264\",\n            \"-preset\", self.config.render.preset, \"-crf\", str(self.config.render.crf),\n            \"-pix_fmt\", self.config.render.pixel_format, \"-r\", str(self.config.render.fps),\n            str(clip_path),\n        ])\n        return clip_path\n\n    def concatenate(self, clips: list[Path], output_path: Path) -> Path:\n        concat_file = output_path.with_suffix(\".txt\")\n        concat_file.write_text(\"\\n\".join(f\"file '{path.as_posix()}'\" for path in clips), encoding=\"utf-8\")\n        run_command([\n            \"ffmpeg\", \"-y\", \"-f\", \"concat\", \"-safe\", \"0\", \"-i\", str(concat_file),\n            \"-c\", \"copy\", \"-fflags\", \"+genpts\", str(output_path),\n        ])\n        return output_path\n\n    def render(self, storyboard: Storyboard, scene_paths: list[Path], audio_path: Path, ass_path: Path, run_dir: Path) -> tuple[Path, RenderReport]:\n        if len(scene_paths) != len(storyboard.scenes):\n            raise ValueError(\"Scene SVG count does not match storyboard scenes.\")\n        video_dir = run_dir / \"video\"\n        video_dir.mkdir(parents=True, exist_ok=True)\n        clips: list[Path] = []\n        for scene, svg_path in zip(storyboard.scenes, scene_paths):\n            png = self.rasterize(svg_path)\n            clips.append(self.make_scene_clip(scene, png))\n        visuals_path = self.concatenate(clips, video_dir / \"visuals.mp4\")\n\n        final_path = video_dir / \"short.mp4\"\n        ass_filter_path = ass_path.as_posix().replace(\"'\", \"\\\\'\").replace(\":\", \"\\\\:\")\n        run_command([\n            \"ffmpeg\", \"-y\", \"-i\", str(visuals_path), \"-i\", str(audio_path),\n            \"-vf\", f\"ass='{ass_filter_path}'\", \"-map\", \"0:v:0\", \"-map\", \"1:a:0\",\n            \"-c:v\", \"libx264\", \"-preset\", self.config.render.preset,\n            \"-crf\", str(self.config.render.crf), \"-pix_fmt\", self.config.render.pixel_format,\n            \"-c:a\", \"aac\", \"-b:a\", self.config.render.audio_bitrate, \"-ar\", \"48000\",\n            \"-movflags\", \"+faststart\", \"-shortest\", str(final_path),\n        ])\n        report = self.validate(final_path)\n        atomic_write_json(video_dir / \"render_report.json\", report)\n        if self.config.render.validate_output and not report.passed:\n            raise RuntimeError(\"Final video failed validation: \" + \"; \".join(report.checks))\n        return final_path, report\n\n    def validate(self, video_path: Path) -> RenderReport:\n        probe = run_command([\n            \"ffprobe\", \"-v\", \"error\", \"-show_streams\", \"-show_format\", \"-of\", \"json\", str(video_path),\n        ])\n        data = json.loads(probe.stdout)\n        streams = data.get(\"streams\", [])\n        video_stream = next((stream for stream in streams if stream.get(\"codec_type\") == \"video\"), {})\n        audio_stream = next((stream for stream in streams if stream.get(\"codec_type\") == \"audio\"), {})\n        width = int(video_stream.get(\"width\") or 0)\n        height = int(video_stream.get(\"height\") or 0)\n        frame_rate = str(video_stream.get(\"avg_frame_rate\") or \"0/1\")\n        numerator, denominator = [float(value) for value in frame_rate.split(\"/\")]\n        fps = numerator / denominator if denominator else 0.0\n        duration = float((data.get(\"format\") or {}).get(\"duration\") or 0.0)\n        checks: list[str] = []\n        if width != self.config.render.width or height != self.config.render.height:\n            checks.append(f\"resolution={width}x{height}\")\n        if abs(fps - self.config.render.fps) > 0.5:\n            checks.append(f\"fps={fps:.2f}\")\n        if not audio_stream:\n            checks.append(\"missing audio stream\")\n        if duration <= 1.0:\n            checks.append(f\"duration={duration:.2f}s\")\n        passed = not checks\n        return RenderReport(\n            video_path=str(video_path), width=width, height=height, fps=round(fps, 3),\n            duration_s=round(duration, 3), video_codec=str(video_stream.get(\"codec_name\") or \"\"),\n            audio_codec=str(audio_stream.get(\"codec_name\") or \"\") or None, has_audio=bool(audio_stream),\n            file_size_bytes=video_path.stat().st_size, render_hash=file_sha256(video_path), passed=passed,\n            checks=checks or [\"resolution, fps, duration, video stream, and audio stream passed\"],\n        )\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 16. SEO metadata

Upload-ready YouTube metadata (title, description, hashtags, tags, filename) — LLM-authored with a deterministic fallback, then hard-clamped to the configured limits and cleared of misleading superlatives.

In [ ]:
# Section 16 — SEO metadata generator
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "seo.py": "from __future__ import annotations\n\nfrom pathlib import Path\n\nfrom .config import StudioConfig\nfrom .hashing import atomic_write_json, hash_value, read_json, slugify\nfrom .llm import LLMClient\nfrom .schemas import ResearchBundle, ScriptPackage, SEOPackage\n\n\nclass SEOGenerator:\n    \"\"\"Upload-ready YouTube metadata. LLM-authored with a deterministic fallback,\n    then hard-clamped to the configured title/hashtag/tag limits.\"\"\"\n\n    def __init__(self, config: StudioConfig, llm: LLMClient, cache_root: Path):\n        self.config = config\n        self.llm = llm\n        self.cache_dir = cache_root / \"metadata\"\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n\n    def _fallback(self, script: ScriptPackage, research: ResearchBundle) -> SEOPackage:\n        title = (script.topic.strip().rstrip(\"?\") + \"?\")[: self.config.seo.max_title_chars]\n        hashtags = [\"#WhatIf\", \"#Science\", \"#Animation\", \"#Explained\", \"#Shorts\"]\n        tags = [\n            \"what if\", \"science explained\", \"scientific animation\", \"educational shorts\",\n            \"science simulation\", script.topic.lower(), \"flat vector animation\", \"science facts\",\n        ]\n        description = (\n            f\"{script.hook}\\n\\nThis scientific animation explores {script.topic.lower()} \"\n            f\"using source-backed research and a hypothetical simulation. \"\n            f\"The outcome is educational, not a prediction.\\n\\n\" + \" \".join(hashtags)\n        )\n        return SEOPackage(\n            title=title, description=description, hashtags=hashtags,\n            tags=tags[: self.config.seo.tag_count], filename=slugify(title) + \".mp4\", seo_score=78,\n        )\n\n    def generate(self, script: ScriptPackage, research: ResearchBundle, force_refresh: bool = False) -> SEOPackage:\n        key = hash_value({\"script\": script.script_hash, \"research\": research.research_hash})\n        cache_path = self.cache_dir / f\"{key}.json\"\n        if cache_path.exists() and not force_refresh:\n            return SEOPackage.model_validate(read_json(cache_path))\n        prompt = f\"\"\"\nCreate upload metadata for an English scientific YouTube Short.\nTopic: {script.topic}\nHook: {script.hook}\nEnding: {script.ending}\nResearch summary: {research.summary}\n\nReturn JSON:\n{{\n  \"title\": \"maximum {self.config.seo.max_title_chars} characters\",\n  \"description\": \"clear, intriguing, non-misleading description\",\n  \"hashtags\": [\"exactly {self.config.seo.hashtag_count} focused hashtags\"],\n  \"tags\": [\"up to {self.config.seo.tag_count} focused tags\"],\n  \"filename\": \"safe-file-name.mp4\",\n  \"seo_score\": 0\n}}\nAvoid miracle, guaranteed, shocking, or 100% safe.\n\"\"\"\n        try:\n            raw = self.llm.generate_json(\n                \"You write accurate YouTube metadata for science animations. Return JSON only.\",\n                prompt, cache_namespace=\"seo\", max_new_tokens=900, force_refresh=force_refresh,\n            )\n            package = SEOPackage.model_validate(raw)\n        except Exception:\n            package = self._fallback(script, research)\n        package = package.model_copy(update={\n            \"title\": package.title[: self.config.seo.max_title_chars],\n            \"hashtags\": package.hashtags[: self.config.seo.hashtag_count],\n            \"tags\": package.tags[: self.config.seo.tag_count],\n            \"filename\": slugify(package.filename.removesuffix(\".mp4\")) + \".mp4\",\n        })\n        atomic_write_json(cache_path, package)\n        return package\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 17. Full pipeline export and orchestration

- **`topic.py`** — the three topic modes: manual, keyword expansion (rank +
  select), and auto discovery from trending science headlines.
- **`exporter.py`** — writes the canonical run layout and a zipped copy.
- **`pipeline.py`** — `StudioPipeline`, the single entry point wiring every stage
  from topic to validated MP4. This is the exact object a FastAPI worker calls.

In [ ]:
# Section 17 — topic resolver, exporter, full pipeline
# Idempotent: writing these modules again is safe and cheap.
from pathlib import Path

MODULE_SOURCES = {
    "topic.py": "from __future__ import annotations\n\nimport json\nfrom enum import Enum\n\nfrom .llm import LLMClient\nfrom .search import SearchService\n\n\nclass TopicMode(str, Enum):\n    MANUAL = \"manual\"\n    KEYWORD = \"keyword\"\n    AUTO = \"auto\"\n\n\nclass TopicResolver:\n    \"\"\"Three input modes:\n\n    * ``manual``  \u2014 use the given topic verbatim.\n    * ``keyword`` \u2014 expand a keyword into ranked \"What if\" candidates, pick the best.\n    * ``auto``    \u2014 discover trending science headlines, convert, rank, and select.\n    \"\"\"\n\n    def __init__(self, llm: LLMClient, search: SearchService):\n        self.llm = llm\n        self.search = search\n\n    def manual(self, topic: str) -> str:\n        if not topic.strip():\n            raise ValueError(\"Manual topic cannot be empty.\")\n        return topic.strip()\n\n    def keyword(self, keyword: str) -> tuple[str, list[dict]]:\n        prompt = (\n            f'Expand the scientific keyword \"{keyword}\" into 8 advertiser-friendly What If topics. '\n            \"Rank by visual clarity, scientific depth, curiosity, and flat-SVG feasibility. \"\n            \"Avoid medical advice, animal harm, victim-focused disasters, and unsupported claims. \"\n            'Return JSON: {\"candidates\":[{\"topic\":\"What if ...?\",\"score\":0.0,\"reason\":\"...\"}],\"selected\":\"What if ...?\"}'\n        )\n        result = self.llm.generate_json(\n            \"You are a scientific topic editor. Return JSON only.\",\n            prompt, cache_namespace=\"topic-keyword\", max_new_tokens=1000,\n        )\n        candidates = result.get(\"candidates\") or []\n        selected = str(result.get(\"selected\") or \"\")\n        if not selected and candidates:\n            selected = str(max(candidates, key=lambda item: float(item.get(\"score\", 0))).get(\"topic\"))\n        return selected or f\"What if {keyword} changed the world?\", candidates\n\n    def automatic(self) -> tuple[str, list[dict]]:\n        titles = self.search.discover_trending_titles(24)\n        prompt = (\n            \"Turn these recent science and technology headlines into safe, timeless What If animation topics. \"\n            \"Rank by scientific value, visual clarity, curiosity, and reusable SVG feasibility.\\n\\n\"\n            f\"{json.dumps(titles, ensure_ascii=False)}\\n\\n\"\n            'Return JSON: {\"candidates\":[{\"topic\":\"What if ...?\",\"score\":0.0,\"source_headline\":\"...\",\"reason\":\"...\"}],\"selected\":\"What if ...?\"}'\n        )\n        result = self.llm.generate_json(\n            \"You are a scientific trend editor. Avoid sensationalism. Return JSON only.\",\n            prompt, cache_namespace=\"topic-auto\", max_new_tokens=1300,\n        )\n        candidates = result.get(\"candidates\") or []\n        selected = str(result.get(\"selected\") or \"\")\n        if not selected and candidates:\n            selected = str(max(candidates, key=lambda item: float(item.get(\"score\", 0))).get(\"topic\"))\n        return selected or \"What if Earth stopped rotating for one second?\", candidates\n\n    def resolve(self, mode: str, value: str = \"\") -> tuple[str, list[dict]]:\n        parsed = TopicMode(mode)\n        if parsed is TopicMode.MANUAL:\n            return self.manual(value), []\n        if parsed is TopicMode.KEYWORD:\n            return self.keyword(value)\n        return self.automatic()\n",
    "exporter.py": "from __future__ import annotations\n\nimport shutil\nfrom pathlib import Path\n\nfrom .hashing import atomic_write_json\nfrom .schemas import AudioTimeline, ResearchBundle, RunManifest, ScriptPackage, SEOPackage, Storyboard, ValidationReport\n\n\nclass Exporter:\n    \"\"\"Writes the per-run deliverables in the canonical layout:\n\n    run_dir/\n      research/{research.json, validation.json}\n      scripts/script.json\n      storyboards/storyboard.json\n      audio/timeline.json          metadata/seo.json\n      assets/<asset_id>.svg        scenes/<scene_id>.svg\n      video/short.mp4              captions/{captions.srt, captions.ass}\n      manifest.json  (+ a zipped copy of the whole run)\n    \"\"\"\n\n    def __init__(self, project_root: Path):\n        self.project_root = project_root\n\n    def export_structured(\n        self,\n        run_dir: Path,\n        research: ResearchBundle,\n        validation: ValidationReport,\n        script: ScriptPackage,\n        storyboard: Storyboard,\n        timeline: AudioTimeline,\n        seo: SEOPackage,\n    ) -> dict[str, str]:\n        research_dir = run_dir / \"research\"\n        scripts_dir = run_dir / \"scripts\"\n        storyboard_dir = run_dir / \"storyboards\"\n        metadata_dir = run_dir / \"metadata\"\n        for directory in (research_dir, scripts_dir, storyboard_dir, metadata_dir):\n            directory.mkdir(parents=True, exist_ok=True)\n        files = {\n            \"research\": research_dir / \"research.json\",\n            \"validation\": research_dir / \"validation.json\",\n            \"script\": scripts_dir / \"script.json\",\n            \"storyboard\": storyboard_dir / \"storyboard.json\",\n            \"audio_timeline\": run_dir / \"audio\" / \"timeline.json\",\n            \"seo\": metadata_dir / \"seo.json\",\n        }\n        atomic_write_json(files[\"research\"], research)\n        atomic_write_json(files[\"validation\"], validation)\n        atomic_write_json(files[\"script\"], script)\n        atomic_write_json(files[\"storyboard\"], storyboard)\n        atomic_write_json(files[\"audio_timeline\"], timeline)\n        atomic_write_json(files[\"seo\"], seo)\n        return {key: str(path) for key, path in files.items()}\n\n    def copy_assets(self, run_dir: Path, asset_paths: dict[str, Path]) -> dict[str, str]:\n        assets_dir = run_dir / \"assets\"\n        assets_dir.mkdir(parents=True, exist_ok=True)\n        output: dict[str, str] = {}\n        for asset_id, source in asset_paths.items():\n            destination = assets_dir / f\"{asset_id}.svg\"\n            shutil.copy2(source, destination)\n            output[asset_id] = str(destination)\n        return output\n\n    def write_manifest(self, run_dir: Path, manifest: RunManifest) -> Path:\n        path = run_dir / \"manifest.json\"\n        atomic_write_json(path, manifest)\n        return path\n\n    def zip_run(self, run_dir: Path) -> Path:\n        return Path(shutil.make_archive(str(run_dir), \"zip\", root_dir=run_dir))\n",
    "pipeline.py": "from __future__ import annotations\n\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nfrom tqdm.auto import tqdm\n\nfrom .audio import AudioMixer, VoiceoverEngine\nfrom .cache import AssetCache\nfrom .captions import CaptionGenerator\nfrom .composer import SceneComposer\nfrom .config import StudioConfig\nfrom .exporter import Exporter\nfrom .hardware import detect_hardware\nfrom .hashing import file_sha256, slugify\nfrom .llm import LLMClient\nfrom .logging_utils import configure_logging\nfrom .renderer import VideoRenderer\nfrom .research import ResearchService\nfrom .scene_planner import ScenePlanner\nfrom .schemas import RunManifest\nfrom .script_generator import ScriptGenerator\nfrom .search import SearchService\nfrom .seo import SEOGenerator\nfrom .storyboard_generator import StoryboardGenerator\nfrom .topic import TopicResolver\n\n\nclass StudioPipeline:\n    \"\"\"Full end-to-end orchestration (Phase 1 + Phase 2):\n\n    Topic -> Research -> Validation -> Script -> Storyboard -> Scene plan\n    -> SVG assets (cached) -> Static scene composition\n    -> Voice-over -> Captions -> Music/SFX mix\n    -> Animated 1080x1920 MP4 (ffprobe-validated) -> SEO -> Manifest + zip.\n\n    Every stage is also usable on its own; this class is a thin, testable\n    coordinator that drops straight into a FastAPI background job.\n    \"\"\"\n\n    def __init__(self, config: StudioConfig):\n        self.config = config\n        self.root = Path(config.project.root).resolve()\n        self.cache_root = self.root / \"cache\"\n        self.logger = configure_logging(\"autostudio.pipeline\")\n        self.llm = LLMClient(config, self.cache_root / \"llm\")\n        self.search = SearchService(config, self.cache_root)\n        self.topic_resolver = TopicResolver(self.llm, self.search)\n        self.research = ResearchService(config, self.llm, self.cache_root / \"research\")\n        self.script = ScriptGenerator(config, self.llm, self.cache_root / \"scripts\")\n        self.storyboard = StoryboardGenerator(config, self.llm, self.cache_root / \"storyboards\")\n        self.planner = ScenePlanner(config)\n        self.asset_cache = AssetCache(config, self.cache_root)\n        self.composer = SceneComposer(config)\n        self.voice = VoiceoverEngine(config, self.cache_root)\n        self.captions = CaptionGenerator(config)\n        self.audio_mixer = AudioMixer(config)\n        self.renderer = VideoRenderer(config, self.cache_root)\n        self.seo = SEOGenerator(config, self.llm, self.cache_root)\n        self.exporter = Exporter(self.root)\n\n    def run(self, mode: str, value: str = \"\", *, force_refresh: bool = False, strict_validation: bool = False) -> dict:\n        topic, candidates = self.topic_resolver.resolve(mode, value)\n        run_id = datetime.now().strftime(\"%Y%m%d_%H%M%S\") + \"_\" + slugify(topic, 48)\n        run_dir = self.root / \"output\" / run_id\n        run_dir.mkdir(parents=True, exist_ok=True)\n        self.logger.info(\"Run %s | topic=%s\", run_id, topic)\n        progress = tqdm(total=14, desc=\"Scientific animation studio\", unit=\"stage\")\n\n        # --- Phase 1: research -> storyboard -> static scenes --------------- #\n        sources = self.search.search_topic(topic, force_refresh)\n        progress.update(1)\n        research, validation = self.research.collect(topic, sources, force_refresh)\n        progress.update(1)\n        if strict_validation and not validation.passed:\n            progress.close()\n            raise RuntimeError(\"Scientific validation failed in strict mode.\")\n\n        script = self.script.generate(research, validation, force_refresh)\n        progress.update(1)\n        storyboard = self.planner.plan(self.storyboard.generate(script, research, force_refresh))\n        progress.update(1)\n\n        asset_paths: dict[str, Path] = {}\n        asset_hashes: dict[str, str] = {}\n        for requirement in tqdm(storyboard.asset_catalog, desc=\"SVG assets\", leave=False):\n            path, metadata = self.asset_cache.get_or_create(requirement, topic)\n            asset_paths[requirement.asset_id] = path\n            asset_hashes[requirement.asset_id] = metadata.asset_hash\n        progress.update(1)\n\n        scene_dir = run_dir / \"scenes\"\n        scene_dir.mkdir(parents=True, exist_ok=True)\n        scene_paths: list[Path] = []\n        for index, scene in enumerate(tqdm(storyboard.scenes, desc=\"Static scene composition\", leave=False), start=1):\n            scene_paths.append(self.composer.compose_scene(\n                scene, asset_paths, scene_dir / f\"scene{index:02d}.svg\",\n                storyboard.canvas_width, storyboard.canvas_height,\n            ))\n        progress.update(1)\n\n        contact_sheet = self.composer.compose_contact_sheet(scene_paths, run_dir / \"storyboard_contact_sheet.svg\")\n        preview_html = self.composer.compose_preview_html(scene_paths, run_dir / \"preview.html\")\n        progress.update(1)\n\n        # --- Phase 2: voice -> captions -> mix -> video -------------------- #\n        timeline, storyboard = self.voice.build_timeline(storyboard, run_dir)\n        progress.update(1)\n        srt_path, ass_path, caption_segments = self.captions.write(timeline, run_dir)\n        progress.update(1)\n        final_audio = self.audio_mixer.mix(timeline, run_dir)\n        progress.update(1)\n\n        # Re-compose scenes so their durations match the synthesised audio.\n        scene_paths = [\n            self.composer.compose_scene(\n                scene, asset_paths, scene_dir / f\"scene{index:02d}.svg\",\n                storyboard.canvas_width, storyboard.canvas_height,\n            )\n            for index, scene in enumerate(storyboard.scenes, start=1)\n        ]\n        video_path, render_report = self.renderer.render(storyboard, scene_paths, final_audio, ass_path, run_dir)\n        progress.update(1)\n        seo = self.seo.generate(script, research, force_refresh)\n        progress.update(1)\n\n        # --- Export + manifest --------------------------------------------- #\n        files = self.exporter.export_structured(run_dir, research, validation, script, storyboard, timeline, seo)\n        copied_assets = self.exporter.copy_assets(run_dir, asset_paths)\n        files.update({\n            \"contact_sheet\": str(contact_sheet),\n            \"preview_html\": str(preview_html),\n            \"scenes_directory\": str(scene_dir),\n            \"assets_directory\": str(run_dir / \"assets\"),\n            \"voice_track\": timeline.voice_track,\n            \"final_audio\": str(final_audio),\n            \"captions_srt\": str(srt_path),\n            \"captions_ass\": str(ass_path),\n            \"video\": str(video_path),\n            \"render_report\": str(run_dir / \"video\" / \"render_report.json\"),\n        })\n\n        manifest = RunManifest(\n            run_id=run_id, topic=topic, mode=mode,\n            created_at=datetime.now(timezone.utc).isoformat(),\n            project_root=str(self.root), run_directory=str(run_dir),\n            hardware=detect_hardware().to_dict(),\n            research_hash=research.research_hash, script_hash=script.script_hash,\n            storyboard_hash=storyboard.storyboard_hash, asset_hashes=asset_hashes,\n            files=files, validation_passed=validation.passed,\n            estimated_duration_s=storyboard.estimated_duration_s,\n            warnings=[issue.message for issue in validation.issues],\n            video_hash=file_sha256(video_path),\n            render_report=render_report.model_dump(mode=\"json\"),\n        )\n        manifest_path = self.exporter.write_manifest(run_dir, manifest)\n        archive_path = self.exporter.zip_run(run_dir)\n        progress.update(1)\n        progress.close()\n\n        if self.config.llm.unload_after_pipeline:\n            self.llm.unload()\n\n        return {\n            \"topic\": topic, \"topic_candidates\": candidates, \"sources\": sources,\n            \"research\": research, \"validation\": validation, \"script\": script,\n            \"storyboard\": storyboard, \"asset_paths\": asset_paths, \"copied_assets\": copied_assets,\n            \"scene_paths\": scene_paths, \"contact_sheet\": contact_sheet, \"preview_html\": preview_html,\n            \"timeline\": timeline, \"caption_segments\": caption_segments,\n            \"captions_srt\": srt_path, \"captions_ass\": ass_path, \"final_audio\": final_audio,\n            \"video_path\": video_path, \"render_report\": render_report, \"seo\": seo,\n            \"manifest\": manifest, \"manifest_path\": manifest_path, \"archive_path\": archive_path,\n            \"run_dir\": run_dir,\n        }\n",
}
for _name, _src in MODULE_SOURCES.items():
    _path = PACKAGE_ROOT / _name
    _path.write_text(_src, encoding='utf-8')
    print('wrote', _path.relative_to(PROJECT_ROOT))

## 18. Topic input and execution settings

- `manual` — `TOPIC_VALUE` is used verbatim.
- `keyword` — `TOPIC_VALUE` (e.g. `"Black Hole"`) is expanded, ranked, selected.
- `auto` — trending scientific topics are discovered, ranked, and selected.

In [ ]:
TOPIC_MODE = "manual"   # "manual" | "keyword" | "auto"
TOPIC_VALUE = "What if Earth suddenly stopped rotating?"

FORCE_REFRESH = False                 # True bypasses all caches for this run.
STRICT_SCIENTIFIC_VALIDATION = False  # True aborts if validation fails.

print("Mode:", TOPIC_MODE)
print("Input:", TOPIC_VALUE)

## 19. Run the complete pipeline

Executes Topic → … → validated MP4 and writes every artifact under `output/<run_id>/`. First run downloads Qwen and synthesises audio, so it takes a few minutes; later runs reuse the caches.

In [ ]:
from autostudio.config import load_config
from autostudio.pipeline import StudioPipeline

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
pipeline = StudioPipeline(config)

result = pipeline.run(
    TOPIC_MODE, TOPIC_VALUE,
    force_refresh=FORCE_REFRESH,
    strict_validation=STRICT_SCIENTIFIC_VALIDATION,
)

report = result["render_report"]
print("\nTopic:            ", result["topic"])
print("Sources found:    ", len(result["sources"]))
print("Validation passed:", result["validation"].passed, "| coverage:", result["validation"].coverage_score)
print("Scenes:           ", len(result["storyboard"].scenes), "| unique assets:", len(result["asset_paths"]))
print("Video:            ", result["video_path"])
print("  ", report.width, "x", report.height, "@", report.fps, "fps |",
      round(report.duration_s, 1), "s |", report.video_codec, "+", report.audio_codec,
      "| validation:", "PASSED" if report.passed else "FAILED")
print("Downloadable zip: ", result["archive_path"])

## 20. Storyboard and final video preview

In [ ]:
from IPython.display import HTML, SVG, Video, display

print("Final video:")
display(Video(str(result["video_path"]), embed=True, width=360))

print("Composed scenes:")
for scene_path in result["scene_paths"]:
    display(SVG(filename=str(scene_path)))

## 21. Inspect structured outputs

In [ ]:
from IPython.display import JSON, display

display(JSON({
    "research_summary": result["research"].summary,
    "script_beats": [b.model_dump(mode="json") for b in result["script"].beats],
    "asset_catalog": [a.model_dump(mode="json") for a in result["storyboard"].asset_catalog],
    "seo": result["seo"].model_dump(mode="json"),
    "render_report": result["render_report"].model_dump(mode="json"),
}))

## 22. Download the complete video project

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(result["archive_path"]))
else:
    print("Run archive:", result["archive_path"])

## Cache inspection

Proof of reuse across the asset, raster, and scene-clip caches.

In [ ]:
import json

asset_cache_dir = PROJECT_ROOT / "cache" / "assets"
for meta_path in sorted(asset_cache_dir.glob("*.metadata.json"))[:6]:
    meta = json.loads(meta_path.read_text())
    print(f"  {meta['asset_type']:<14} reuse={meta['reuse_counter']:<3} hash={meta['asset_hash'][:12]}")

for name in ["raster", "scene_clips", "audio"]:
    d = PROJECT_ROOT / "cache" / name
    n = len(list(d.glob("*"))) if d.exists() else 0
    print(f"cache/{name}: {n} cached files")

## 23. Offline full-render smoke test (no internet, no LLM)

Exercises the entire media path — SVG asset → scene SVG → raster → motion clip →
captions → synthetic voice + music/SFX mix → muxed MP4 → `ffprobe` validation —
with **no network and no model**. This is the institutional-grade CI gate: it
proves the render toolchain end-to-end before any GPU/model time is spent.

In [ ]:
from xml.etree import ElementTree as ET

from autostudio.audio import AudioMixer, run_command
from autostudio.cache import AssetCache
from autostudio.captions import CaptionGenerator
from autostudio.composer import SceneComposer
from autostudio.config import load_config
from autostudio.renderer import VideoRenderer
from autostudio.schemas import (
    AudioClip, AudioTimeline, AssetRequirement, DashboardSpec, Scene, SceneObject, Storyboard,
)

cfg = load_config(PROJECT_ROOT / "config" / "config.yaml")
TEST_DIR = PROJECT_ROOT / "tests" / "full_render_smoke"
TEST_DIR.mkdir(parents=True, exist_ok=True)

requirements = [
    AssetRequirement(asset_id="earth-test", asset_type="planet", label="Earth", variant="earth", source_prompt="flat Earth"),
    AssetRequirement(asset_id="arrow-test", asset_type="arrow", label="Motion", source_prompt="blue motion arrow"),
]
asset_cache = AssetCache(cfg, PROJECT_ROOT / "cache")
asset_paths = {r.asset_id: asset_cache.get_or_create(r, "offline smoke test")[0] for r in requirements}
scenes = [
    Scene(scene_id="scene01", duration_s=2.0, narration="Earth is rotating faster than it looks.",
          title="EARTH IS MOVING", motion="zoom_in",
          dashboard=DashboardSpec(experiment_id="EXPERIMENT #001", metric_label="ROTATION", metric_value="1670 km/h"),
          objects=[SceneObject(asset_id="earth-test", x=0.18, y=0.30, width=0.64, height=0.40)],
          asset_requirements=[requirements[0]]),
    Scene(scene_id="scene02", duration_s=2.0, narration="A sudden stop would preserve that sideways momentum.",
          title="MOMENTUM CONTINUES", motion="pan_right", background="dark",
          dashboard=DashboardSpec(experiment_id="EXPERIMENT #002", metric_label="STATUS", metric_value="CRITICAL", severity="critical"),
          objects=[SceneObject(asset_id="earth-test", x=0.08, y=0.33, width=0.42, height=0.32),
                   SceneObject(asset_id="arrow-test", x=0.46, y=0.38, width=0.44, height=0.22)],
          asset_requirements=requirements),
]
storyboard = Storyboard(topic="Offline smoke test", scenes=scenes, asset_catalog=requirements, estimated_duration_s=4.0)
composer = SceneComposer(cfg)
scene_paths = [composer.compose_scene(s, asset_paths, TEST_DIR / f"{s.scene_id}.svg", 1080, 1920) for s in scenes]

voice_path = TEST_DIR / "voice_track.m4a"
run_command(["ffmpeg", "-y", "-f", "lavfi", "-i", "sine=frequency=260:sample_rate=48000:duration=4",
             "-c:a", "aac", "-b:a", "160k", str(voice_path)])
timeline = AudioTimeline(
    clips=[AudioClip(scene_id="scene01", text=scenes[0].narration, path=str(voice_path), start_s=0, end_s=2, duration_s=2, provider="synthetic-test", audio_hash="t1"),
           AudioClip(scene_id="scene02", text=scenes[1].narration, path=str(voice_path), start_s=2, end_s=4, duration_s=2, provider="synthetic-test", audio_hash="t2")],
    voice_track=str(voice_path), duration_s=4, timeline_hash="offline-test")
final_audio = AudioMixer(cfg).mix(timeline, TEST_DIR)
_, ass_path, _ = CaptionGenerator(cfg).write(timeline, TEST_DIR)

video_path, report = VideoRenderer(cfg, PROJECT_ROOT / "cache").render(storyboard, scene_paths, final_audio, ass_path, TEST_DIR)

assert report.passed, report.checks
assert report.width == 1080 and report.height == 1920
assert report.has_audio and video_path.exists() and video_path.stat().st_size > 10000
for p in scene_paths:
    assert ET.parse(p).getroot().tag.split("}")[-1] == "svg"

print("FULL-RENDER SMOKE TEST PASSED")
print(report.model_dump(mode="json"))

## Production integration contract

The generated `src/autostudio/` package is the portable core; it moves without a
rewrite into:

- **FastAPI** — call `StudioPipeline.run()` from a background job; return `manifest.json`.
- **Docker** — install the same pins + system packages and copy `src/autostudio/`.
- **Motion Canvas / Remotion** — consume `storyboard.json` + the SVG asset cache
  for richer object animation; the FFmpeg renderer here is the baseline.
- **YouTube uploader** — upload `video/short.mp4` with `metadata/seo.json` after a
  separate OAuth step.

### Verified guarantees

- deterministic content hashing and cross-run cache reuse (assets, rasters,
  scene clips, voice clips);
- free-only web research with source diversity + numeric-claim validation;
- whitelisted, reproducible, flat-vector SVG assets (no raster/gradient);
- voice-over with Edge TTS → espeak-ng fallback;
- safe-zone SRT + ASS captions; procedural, licence-free music + SFX;
- FFmpeg camera motion, concatenation, caption burn-in, and A/V muxing;
- `ffprobe` validation of resolution, fps, duration, codec, and audio stream;
- a fully offline, network-free full-render smoke test (section 23).

Live Qwen inference, Edge TTS, and public web search need runtime internet; the
deterministic render path is fully testable offline.